# BTC Orderbook LSTM — run.008 (maker-entry simulation + entry-delay stress)

**Why:** run.007 failed its gate in the pre-registered failure mode #2 — *"lift good, sim ≤ 0, TP-grid all ≤ 0 → fees eat the touches; the remaining lever is ENTRY price"*. The ranking signal was the strongest in the project (h6 @ 0.1 % lift 9.4–9.8×, 4/4 folds; seed-unanimity up6 hit **49.7 %**), but the taker-entry arithmetic is closed: a hit pays θ − 7 bp = +13 bp, the average non-hit costs ≈ −18.4 bp, so breakeven hit ≈ **58.5 %**. At a full-maker round trip (4 bp) a hit pays +16 bp and breakeven drops to ≈ **49 %** — within one hit-rate point of what the heads already deliver. run.008 measures whether that point survives an **honest fill model**. It also runs the promised **+1-bar entry-delay stress** on run.007's zero-latency taker assumption.

**Training is byte-frozen from run.007** (cells 4–9: features, load, folds, model, training loop — identical bytes; cell 9's npz filename is the only training-side change). Cell 10 reprints run.007's taker-entry tables *unchanged* (its strings still say "run.007" — intentional: it is the reproduction/baseline, and its `sim_exit`/`trigger_stats`/rolling-τ machinery is reused). Everything new lives in cell 10M and the gate.

**What is new (cell 10M):**

1. **Maker-entry fill model** — post a limit at `close[e]·(1 − sgn·δ·θ)` after `ENTRY_DELAY` bars; it fills **at the limit** on the first of the next `ENTRY_WAIT = 2` bars whose *close* trades through it (the same conservative convention as run.007's TP fill); unfilled → cancel, **no trade**. After the fill: TP limit maker at entry·(1+sgn·θ), SL taker at the breaching bar's actual close (the fill bar's own close counts), else time-stop taker at `close[e+h]` — the horizon stays anchored at the **trigger**, so waiting for a fill consumes move window and the sim pays for it. Wait = 2 bars is pre-registered: run.007's h2 signal (t = +12.9) says the pullback lives at the 30 s scale, and 2 bars leaves ≥ 4 of the 6-bar window after the fill.
2. **Fill-rate-honest accounting.** Three numbers must be read together: `sim` (net bps over *filled* trades — the CI belongs here), `fill%` (a resting limit misses exactly the moves that leave immediately), and `EV/trig` (per-signal expectation with unfilled = 0 — directly comparable to cell 10's taker column). Plus the adverse-selection diagnostic: gross endpoint move and hit rate of the **unfilled** triggers (`missed`, `hit_m`). If `hit_m ≫ hit_f`, the limit order filters out the winners — the failure mode this design must expose, not hide.
3. **h2-timed hybrid (exploratory, not gated)** — run.007's strongest-ever regression signal (IC_h2 + 0.038, t = +12.9, fast-dying mean reversion) is useless standalone against fees but is a natural *execution* signal: triggers where `sgn·pred_h2 > 0` (the move is leaving now) enter taker immediately; the rest post the maker limit into the predicted pullback.
4. **Entry-delay stress** — taker entry shifted to `close[e+1]` (implemented exactly as `sim_exit(e+1, side, h−1, …)`, time-stop unchanged at `e+h`), and a maker `delay=1` variant.

**Pre-registered decision rule (GATE, cell 11)** — h6 heads, 0.1 % rolling ens, maker primary config (wait 2, δ = 0, delay 0), TP = SL = 1.0·θ, either side:
- **A.** pooled maker-sim net (filled trades) > 0 **and** day-boot 95 % CI excludes 0;
- **B.** pooled hit lift ≥ 3× **and** per-fold lift ≥ 2× in ≥ 3/4 folds (unchanged; on all triggers);
- **C.** pooled fill rate ≥ 25 % **and** per-fold maker sim > 0 in ≥ 3/4 folds (run.007's fold-2 collapse must fail here, not hide in the pool);
- **D.** neighbouring rates (0.01 %, 1 %) pooled maker sim > 0.

**Caveats:**
- Fills are decided on 15 s bar **closes** of the mid: a touch of the limit without a close through it does not fill (conservative), but queue position and partial fills are ignored (optimistic), and mid ≠ trade price. No slippage model. A close that gaps through limit and stop in one bar fills and immediately stops — counted.
- The maker TP exit and maker entry both assume the limit is filled whenever price closes through it — adverse queue dynamics at exactly these moments are the first thing a freqtrade cross-check must probe if the gate passes.
- If `btc_data.tar.xz` still lacks out3/out4 files this is again a **v1-feature run** (6th in a row — rebuild the tar with `tar cJf btc_data.tar.xz out*.csv.gz` and verify with `tar tf … | grep -c '^out[34]'`). The schema-3/4 retest needs ≥ 45 collector-days of out3/out4 (started 2026-07-07 → ready ~late August).

Expected runtime on a T4: ≈ run.007 (identical training; the maker sim is vectorised numpy).

## Setup
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order


In [ ]:
# ── Cell 1: Install and imports ───────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch',
                'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'scipy'],
               check=True)

import os, glob, gc, math, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# ── Cell 2: (optional) data transfer helpers — uncomment what you need ────
#!apt install sshpass -y && sshpass -p XXXXXXXXXX sftp -r -o "StrictHostKeyChecking no" scpuser@155.207.120.2:btc_data.tar.xz .
#from google.colab import drive
#drive.mount('/content/drive')
#!mv btc_data.tar.xz /content/drive/MyDrive/
#!ls /content/drive/MyDrive/ -l
#drive.flush_and_unmount()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cd ~ && mkdir -p btc_lstm_v2
!cd ~ && tar xJf /content/drive/MyDrive/btc_data.tar.xz

drive.flush_and_unmount()


In [ ]:
# ── Cell 3: Config ─────────────────────────────────────────────────────────

# ── CONFIGURE THESE ───────────────────────────────────────────────────────
DATA_DIR   = '/root/btc_data'
OUTPUT_DIR = '/root/btc_lstm_run008'

WINDOW_SEC  = 15          # bar size produced by the collector
HORIZONS    = [2, 3, 4, 5, 6, 8]
                          # run.007: 30s–2min. h8 kept as the run.006 anchor
                          # (same θ → lup_8/ldn_8 directly comparable); h40/240
                          # dropped — run.006 showed the θ-touch reverts by bar
                          # 40, and MAX_H 240→8 stops the forward-contiguity
                          # check from eating samples near gaps.
H_TRADE     = 8           # importance sampling / Huber weighting / reference IC
                          # follow the least-noisy kept horizon (run.004's
                          # t=+4.8 short-horizon edge was measured at h8)
VOL_WINDOW  = 240         # 1h rolling window for target normalisation
TARGET_CLIP = 5.0         # clip |target| at 5 sigma

SEQ_LEN      = 64         # 16 min of context
HIDDEN_DIM   = 128        # SMALL on purpose — capacity was cleared twice (run.002/003)
NUM_LAYERS   = 2
DROPOUT      = 0.3
LR           = 1e-3
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 2048
EPOCHS       = 12
WARMUP       = 2          # linear warmup epochs before cosine decay
PATIENCE     = 5          # early stop on val AP of the two H_SEL opportunity heads
EPOCH_SAMPLE_CAP = 600_000  # train sequences drawn per epoch (weighted, w/ replacement)

# ── run.003 (kept): focus training on directional bars ──
SAMPLE_W_BASE = 0.25      # epoch-draw probability ∝ SAMPLE_W_BASE + |y_trade|
LOSS_W_ALPHA  = 1.0       # per-sample Huber weight = min(1 + α·|y_trade|, cap)
LOSS_W_CAP    = 4.0

# ── run.004 (kept): multi-seed evaluation ──
SEEDS = [0, 1, 2]         # per-fold; test score/prob = seed-ensemble mean
MIN_DAY_SAMPLES = 500     # a test day needs this many samples to enter daily-IC stats

N_FOLDS         = 4       # walk-forward folds over the last 60% of the window
TEST_START_FRAC = 0.40    # first 40% is never tested (training warmup)
VAL_FRAC        = 0.12    # tail of each training range used for early stopping
COVERAGE_MIN    = 0.60    # features with lower non-NaN coverage are excluded

MAKER_FEE = 0.0002        # per side (Binance USDT-M futures maker)
TAKER_FEE = 0.0005        # per side

# ── run.006 (kept): schema-3/4 era handling ──
V3_ERA_ONLY  = False      # restrict the analysis window to schema-3+ rows
V3_MARKER    = 'future_ofi_sum'   # first non-NaN value of this = era start
MIN_ERA_DAYS = 45         # abort if less schema-3+ data than this has accumulated
WALL_BAND    = 0.30       # collector WALL_BAND_PCT — wall dist −1 (=absent) maps here

# ── run.007: short-horizon big-move heads ──
H_SEL        = 6          # primary selective horizon (90s), pre-registered.
                          # Rationale: closest kept horizon to run.006's
                          # best-lift h8, θ=20bp base rate stays trainable
                          # (~2–4%), and ~90% of a θ-touch survives to the h6
                          # endpoint (vs ~10% at run.006's h40).
THETA_BY_H   = {2: 0.0015, 3: 0.0015,   # 15 bps: a 20 bps move inside 30–45s
                4: 0.0020, 5: 0.0020,   #   is near-unreachable (base ≲0.3%)
                6: 0.0020, 8: 0.0020}   # 20 bps = 2× taker RT = run.006's θ
POS_W_CAP    = 20.0       # BCE pos_weight = neg/pos per head, capped here
BCE_W        = 1.0        # classification loss weight vs the (frozen-form) Huber part

# ── run.007: TP/SL exit simulation (run.006's #1 lever, untested there) ──
TP_MULT      = 1.0        # primary take-profit = TP_MULT·θ_h (limit, maker exit)
SL_MULT      = 1.0        # primary stop = SL_MULT·θ_h (taker exit at the
                          # breaching bar's CLOSE — gap-through conservative)
TP_SL_GRID   = [(1.0, 1.0), (1.0, 0.5), (1.0, np.inf), (1.5, 1.0), (1.5, 0.5)]
                          # exploratory grid; ONLY (TP_MULT, SL_MULT) is gated

# ── run.007: rolling-causal trigger calibration ──
ROLL_CAL_DAYS = 14        # τ for test day D = (1−rate)-quantile of the scores
                          # seen in [D−14d, D): val seeds the window, already-
                          # seen test scores join it. Deployable (past data
                          # only); fixes run.006's fold-1 zero-trigger pathology.
MIN_CAL_N     = 5000      # thinner window → expand to all past scores

TRIGGER_RATES = [0.0001, 0.001, 0.01]  # nominal trigger rates
PRIMARY_RATE  = 0.001     # pre-registered primary: top-0.1% ≈ a few signals/day
N_BOOT        = 2000      # day-cluster bootstrap draws for net-bps CIs

# ── run.008: maker-entry simulation (run.007's diagnosed remaining lever) ──
ENTRY_WAIT   = 2          # bars the entry limit may rest before cancel
                          # (primary, pre-registered): run.007's h2 signal
                          # (t=+12.9) puts the pullback at the 30s scale, and
                          # 2 bars leaves ≥4 of the 6-bar move window after
                          # the fill
ENTRY_DELTA  = 0.0        # limit offset in θ_h units below (up) / above (dn)
                          # the trigger close; 0 = post AT the signal close
ENTRY_DELAY  = 0          # bars between signal and posting (1 = latency stress)
WAIT_GRID    = [1, 2, 4, 6]        # exploratory
DELTA_GRID   = [0.0, 0.25, 0.5]    # exploratory, ·θ_h
FILL_MIN     = 0.25       # gate C: primary config must fill ≥ this share of
                          # triggers — a maker edge that almost never fills is
                          # a statistical fluke, not a strategy

DRIVE_SAVE_DIR = '/content/drive/MyDrive/btc_lstm_run008'
# ─────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Data dir:   {DATA_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print('θ per horizon: ' + '  '.join(f'h{h}={t*1e4:.0f}bp' for h, t in THETA_BY_H.items()))
print(f'taker RT = {2*TAKER_FEE*1e4:.0f} bps   TP RT (taker in, maker out) = '
      f'{(TAKER_FEE+MAKER_FEE)*1e4:.0f} bps   full-maker RT = '
      f'{2*MAKER_FEE*1e4:.0f} bps')


In [ ]:
# ── Cell 4: Feature pipeline ──────────────────────────────────────────────
SEPARATOR = ';'

COLS_NEEDED_BASE = {
    'spot_datetime', 'future_datetime', 'spot_timestamp', 'future_timestamp',
    'future_bid_close', 'future_ask_close',
    'future_bid_open', 'future_bid_max', 'future_bid_min',
    'future_bid_median', 'future_ask_median',
    'future_spread_open', 'future_spread_max',
    'future_buy_qty', 'future_sell_qty',
    'future_buy_samples', 'future_sell_samples',
    'future_buy_vwap', 'future_sell_vwap', 'future_price_diff',
    'future_bid_liq_0.0_median', 'future_ask_liq_0.0_median',
    'spot_buy_qty', 'spot_sell_qty',
    'spot_bid_close', 'spot_ask_close',
    'opt_open_interest_sample', 'opt_funding_rate_sample',
    'opt_est_funding_rate_sample', 'opt_remaining_time_sample',
    'opt_long_force_exit_qty_sum', 'opt_short_force_exit_qty_sum',
    'future_first_trade_side', 'future_last_trade_side',
    'future_largest_trade_qty', 'future_largest_trade_side',
    'future_buy_count_early', 'future_buy_count_late',
    'future_sell_count_early', 'future_sell_count_late',
    'future_buy_qty_early', 'future_buy_qty_late',
    'future_sell_qty_early', 'future_sell_qty_late',
}
COLS_NEEDED_LIQ_DEEP  = [
    'future_bid_liq_0.04_median', 'future_ask_liq_0.04_median',
    'future_bid_liq_0.05_median', 'future_ask_liq_0.05_median',
    'future_bid_liq_0.06_median', 'future_ask_liq_0.06_median',
]
COLS_NEEDED_LIQ_TOTAL = [
    'future_bid_liq_0.1_median',  'future_ask_liq_0.1_median',
    'future_bid_liq_0.2_median',  'future_ask_liq_0.2_median',
    'future_bid_liq_0.3_median',  'future_ask_liq_0.3_median',
    'future_bid_liq_0.4_median',  'future_ask_liq_0.4_median',
]
# run.006: schema-3/4 collector columns (out3/out4 files; NaN in older files)
COLS_NEEDED_V3 = [
    'future_mid_std', 'future_mid_rv', 'future_mid_flips',
    'future_micro_dev_close', 'future_micro_dev_median',
    'future_ofi_sum',
    'future_add_bid_near_sum', 'future_cancel_bid_near_sum',
    'future_add_ask_near_sum', 'future_cancel_ask_near_sum',
    'future_add_bid_wide_sum', 'future_cancel_bid_wide_sum',
    'future_add_ask_wide_sum', 'future_cancel_ask_wide_sum',
    'future_depth_msg_count', 'future_best_bid_move_count',
    'future_best_ask_move_count',
    'future_bid_depletion_count', 'future_ask_depletion_count',
    'future_buy_size_median', 'future_buy_size_p90',
    'future_sell_size_median', 'future_sell_size_p90',
    'future_bid_wall_qty_median', 'future_bid_wall_qty_max',
    'future_bid_wall_dist_median',
    'future_ask_wall_qty_median', 'future_ask_wall_qty_max',
    'future_ask_wall_dist_median',
    'future_max_batch_buy_qty', 'future_max_batch_sell_qty',
    'future_max_buy_run_qty', 'future_max_sell_run_qty',
    'opt_long_force_exit_cnt_sum', 'opt_short_force_exit_cnt_sum',
    'opt_long_force_exit_notional_sum', 'opt_short_force_exit_notional_sum',
    'opt_force_exit_notional_max',
    'opt_long_short_ratio_sample',
    'opt_eth_mid_open', 'opt_eth_mid_close',
    'schema_version',
]

# run.004: long-timescale context the 64-bar input window cannot see
NEW_FEATURES = [
    'ret_norm_1h', 'ret_norm_4h', 'ret_norm_24h',
    'ma_gap_1h', 'ma_gap_4h', 'ma_gap_24h',
    'vol_ratio_1h_24h', 'range_pos_24h',
    'basis_z_4h', 'basis_mom_1h',
    'dow_sin', 'dow_cos',
]

# run.006: engineered from the schema-3/4 columns. Episodic by nature —
# mostly quiet, occasionally scream — which is the shape a rare-trigger
# model needs (v1's continuous rolling stats are why runs 001–005 came
# out diffuse).
V3_FEATURES = [
    'ofi_z', 'flow_net_near_z', 'cancel_imbal_near',
    'pull_add_bid', 'pull_add_ask', 'flow_net_widex_z',
    'micro_dev_z', 'mid_rv_norm', 'mid_flips_norm', 'msg_rate_norm',
    'depl_imbal', 'depl_total_norm',
    'buy_tail_ratio', 'sell_tail_ratio',
    'liq_cnt_log', 'liq_imbal', 'liq_notional_log', 'liq_notional_max_log',
    'wall_imbal', 'bid_wall_dist', 'ask_wall_dist', 'wall_qty_norm',
    'batch_buy_rel', 'batch_sell_rel', 'sweep_buy_rel', 'sweep_sell_rel',
    'eth_ret_z', 'eth_lead_gap', 'lsr_z',
]

ALL_FEATURES = [
    'book_imbalance', 'flow_imbalance', 'momentum', 'book_imbal_deep',
    'flow_imbal_roll4', 'flow_imbal_roll8', 'book_imbal_roll4', 'book_imbal_roll8',
    'vwap_spread', 'liq_flag', 'stochastic', 'spread_expansion', 'sample_imbalance',
    'flow_agreement', 'oi_change', 'size_imbalance', 'liq_conc_bid', 'liq_conc_ask',
    'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
    'near_funding', 'funding_pressure', 'vol_norm',
    'trade_side_open', 'trade_side_close', 'trade_side_momentum',
    'largest_trade_side', 'largest_trade_rel',
    'buy_accel', 'sell_accel', 'flow_accel', 'buy_count_accel', 'late_imbalance',
] + NEW_FEATURES + V3_FEATURES

# binary opportunity heads: column order is frozen here and reused everywhere
CLS_KEYS = [(h, s) for h in HORIZONS for s in ('up', 'dn')]
CLS_COLS = [f'l{s}_{h}' for h, s in CLS_KEYS]
SEL_UP_J = CLS_KEYS.index((H_SEL, 'up'))
SEL_DN_J = CLS_KEYS.index((H_SEL, 'dn'))


def load_files(files):
    all_needed = (COLS_NEEDED_BASE | set(COLS_NEEDED_LIQ_DEEP)
                  | set(COLS_NEEDED_LIQ_TOTAL) | set(COLS_NEEDED_V3))
    frames = []
    for path in files:
        try:
            hdr = pd.read_csv(path, sep=SEPARATOR, nrows=0)
            usecols = sorted(all_needed & set(hdr.columns))
            if 'spot_datetime' not in usecols:
                print(f'  SKIP {os.path.basename(path)}: missing spot_datetime')
                continue
            df = pd.read_csv(path, sep=SEPARATOR, usecols=usecols, low_memory=False)
            df = df[df['spot_datetime'] != 'spot_datetime']
            frames.append(df)
        except Exception as e:
            print(f'  SKIP {os.path.basename(path)}: {e}')
    if not frames:
        raise RuntimeError('No valid CSV files found.')
    print(f'  Loaded {len(frames)} files')
    return pd.concat(frames, ignore_index=True)


def prepare(df, horizons, vol_window):
    df['dt'] = pd.to_datetime(df['spot_datetime'], errors='coerce')
    df = df.dropna(subset=['dt']).sort_values('dt').reset_index(drop=True)
    n_before = len(df)
    df = df.drop_duplicates(subset='dt', keep='first').reset_index(drop=True)
    if len(df) < n_before:
        print(f'  Dropped {n_before - len(df)} duplicate timestamps')

    skip = {'spot_datetime', 'future_datetime', 'dt'}
    for col in df.columns:
        if col not in skip:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    for col in [c for c in df.columns if c.startswith('opt_')]:
        df[col] = df[col].replace(-1, np.nan)

    if 'future_bid_close' in df.columns and 'future_ask_close' in df.columns:
        df['close_price'] = (df['future_bid_close'] + df['future_ask_close']) / 2.0
    else:
        df['close_price'] = (df['future_bid_median'] + df['future_ask_median']) / 2.0

    # Gap structure: collector restarts / dropped bars break the 15s cadence.
    gap = df['dt'].diff().dt.total_seconds()
    n_breaks = int((gap != WINDOW_SEC).sum()) - 1
    big = gap[gap > 3600]
    print(f'  Cadence breaks: {n_breaks}  (gaps >1h: {len(big)}, largest: {gap.max()/3600:.1f}h)')

    eps = 1e-9
    df['flow_imbalance'] = ((df['future_buy_qty'] - df['future_sell_qty']) /
                            (df['future_buy_qty'] + df['future_sell_qty'] + eps))
    bid0 = df['future_bid_liq_0.0_median']
    ask0 = df['future_ask_liq_0.0_median']
    df['book_imbalance'] = (bid0 - ask0) / (bid0 + ask0 + eps)
    df['momentum'] = df['future_price_diff']

    deep_col = None
    for lvl in ['0.05', '0.04', '0.06']:
        b, a = f'future_bid_liq_{lvl}_median', f'future_ask_liq_{lvl}_median'
        if b in df.columns and a in df.columns:
            deep_col = (b, a); break
    df['book_imbal_deep'] = ((df[deep_col[0]] - df[deep_col[1]]) /
                             (df[deep_col[0]] + df[deep_col[1]] + eps)) if deep_col else np.nan

    df['flow_imbal_roll4'] = df['flow_imbalance'].rolling(4, min_periods=2).mean()
    df['flow_imbal_roll8'] = df['flow_imbalance'].rolling(8, min_periods=4).mean()
    df['book_imbal_roll4'] = df['book_imbalance'].rolling(4, min_periods=2).mean()
    df['book_imbal_roll8'] = df['book_imbalance'].rolling(8, min_periods=4).mean()

    df['vwap_spread'] = (df['future_buy_vwap'] - df['future_sell_vwap']
                         if 'future_buy_vwap' in df.columns else np.nan)
    liq_long  = df.get('opt_long_force_exit_qty_sum',  pd.Series(0, index=df.index))
    liq_short = df.get('opt_short_force_exit_qty_sum', pd.Series(0, index=df.index))
    df['liq_flag'] = ((liq_long + liq_short) > 0).astype(float)

    if all(c in df.columns for c in ['future_bid_max', 'future_bid_min', 'future_bid_close']):
        rng = df['future_bid_max'] - df['future_bid_min']
        df['stochastic'] = np.where(rng > 0, (df['future_bid_close'] - df['future_bid_min']) / rng, 0.5)
    else:
        df['stochastic'] = np.nan

    df['spread_expansion'] = (df['future_spread_max'] - df['future_spread_open']
                              if all(c in df.columns for c in ['future_spread_max', 'future_spread_open'])
                              else np.nan)

    if all(c in df.columns for c in ['future_buy_samples', 'future_sell_samples']):
        bs, ss = df['future_buy_samples'], df['future_sell_samples']
        df['sample_imbalance'] = (bs - ss) / (bs + ss + eps)
    else:
        df['sample_imbalance'] = np.nan

    if all(c in df.columns for c in ['spot_buy_qty', 'spot_sell_qty']):
        spot_flow = ((df['spot_buy_qty'] - df['spot_sell_qty']) /
                     (df['spot_buy_qty'] + df['spot_sell_qty'] + eps))
        df['flow_agreement'] = df['flow_imbalance'] * spot_flow
    else:
        df['flow_agreement'] = np.nan

    df['oi_change'] = df['opt_open_interest_sample'].diff() if 'opt_open_interest_sample' in df.columns else np.nan

    if all(c in df.columns for c in ['future_buy_samples', 'future_sell_samples',
                                      'future_buy_qty', 'future_sell_qty']):
        df['size_imbalance'] = (df['future_buy_qty']  / (df['future_buy_samples']  + eps) -
                                df['future_sell_qty'] / (df['future_sell_samples'] + eps))
    else:
        df['size_imbalance'] = np.nan

    sub_cols = ['future_first_trade_side', 'future_last_trade_side',
                'future_largest_trade_qty', 'future_largest_trade_side',
                'future_buy_count_early', 'future_buy_count_late',
                'future_sell_count_early', 'future_sell_count_late',
                'future_buy_qty_early', 'future_buy_qty_late',
                'future_sell_qty_early', 'future_sell_qty_late']
    if all(c in df.columns for c in sub_cols):
        bqe, bql = df['future_buy_qty_early'],  df['future_buy_qty_late']
        sqe, sql = df['future_sell_qty_early'],  df['future_sell_qty_late']
        fts, lts = df['future_first_trade_side'], df['future_last_trade_side']
        lq,  ls  = df['future_largest_trade_qty'], df['future_largest_trade_side']
        bce, bcl = df['future_buy_count_early'], df['future_buy_count_late']
        total_vol = bqe + bql + sqe + sql
        df['trade_side_open']     = fts
        df['trade_side_close']    = lts
        df['trade_side_momentum'] = lts - fts
        df['largest_trade_side']  = ls
        df['largest_trade_rel']   = lq / (total_vol + eps)
        df['buy_accel']           = bql - bqe
        df['sell_accel']          = sql - sqe
        df['flow_accel']          = (bql - sql) - (bqe - sqe)
        df['buy_count_accel']     = bcl - bce
        df['late_imbalance']      = (bql - sql) / (bql + sql + eps)
    else:
        for c in ['trade_side_open','trade_side_close','trade_side_momentum',
                  'largest_trade_side','largest_trade_rel','buy_accel','sell_accel',
                  'flow_accel','buy_count_accel','late_imbalance']:
            df[c] = np.nan

    deep_bid = next((c for c in ['future_bid_liq_0.4_median','future_bid_liq_0.3_median',
                                  'future_bid_liq_0.2_median','future_bid_liq_0.1_median']
                     if c in df.columns), None)
    deep_ask = next((c for c in ['future_ask_liq_0.4_median','future_ask_liq_0.3_median',
                                  'future_ask_liq_0.2_median','future_ask_liq_0.1_median']
                     if c in df.columns), None)
    df['liq_conc_bid'] = (df['future_bid_liq_0.0_median'] / (df[deep_bid] + eps)
                          if deep_bid else np.nan)
    df['liq_conc_ask'] = (df['future_ask_liq_0.0_median'] / (df[deep_ask] + eps)
                          if deep_ask else np.nan)

    hour, minute = df['dt'].dt.hour, df['dt'].dt.minute
    df['hour_sin']   = np.sin(2 * np.pi * hour   / 24)
    df['hour_cos']   = np.cos(2 * np.pi * hour   / 24)
    df['minute_sin'] = np.sin(2 * np.pi * minute / 60)
    df['minute_cos'] = np.cos(2 * np.pi * minute / 60)

    secs = (hour * 3600 + minute * 60 + df['dt'].dt.second) % (8 * 3600)
    df['near_funding'] = ((8 * 3600 - secs) < 900).astype(float)
    if 'opt_funding_rate_sample' in df.columns:
        funding = df['opt_funding_rate_sample'].replace(-1, np.nan).fillna(0)
        df['funding_pressure'] = funding * df['near_funding']
    else:
        df['funding_pressure'] = np.nan

    df['volatility'] = df['close_price'].diff().abs().rolling(16, min_periods=4).mean()
    df['vol_norm']   = (df['volatility'] /
                        df['volatility'].rolling(5760, min_periods=96).mean()
                       ).replace([np.inf, -np.inf], np.nan)

    # ── run.004: long-timescale context features ─────────────────────────
    # All rolling windows are TIME-based on the datetime index (rolling('24h')),
    # not row-count based: collector restarts land on arbitrary second offsets
    # and gaps would otherwise stretch a "1h" window over more wall time.
    # Everything is causal (past data only) and vol-normalised so the scale is
    # stationary across regimes; clipped at ±8 to bound scaler-era outliers.
    D = 24 * 3600 // WINDOW_SEC              # bars per 24h at full cadence
    lp = pd.Series(np.log(df['close_price'].values), index=df['dt'])

    def past(series, delta, tol_sec=2 * WINDOW_SEC):
        # value of `series` ~delta earlier (nearest bar within tol), aligned back
        p = series.reindex(series.index - delta, method='nearest',
                           tolerance=pd.Timedelta(seconds=tol_sec))
        return pd.Series(p.values, index=series.index)

    r1 = lp.diff()
    r1[gap.values != WINDOW_SEC] = np.nan    # 1-bar return across a gap is not 1-bar
    sigma24 = r1.rolling('24h', min_periods=D // 4).std()

    for name, td, W in [('1h', pd.Timedelta(hours=1),  D // 24),
                        ('4h', pd.Timedelta(hours=4),  D // 6),
                        ('24h', pd.Timedelta(hours=24), D)]:
        unit = sigma24 * np.sqrt(W) + eps
        df[f'ret_norm_{name}'] = ((lp - past(lp, td)) / unit).clip(-8, 8).values
        ma = lp.rolling(td, min_periods=int(W * 0.75)).mean()
        df[f'ma_gap_{name}'] = ((lp - ma) / unit).clip(-8, 8).values

    vol1h = r1.rolling('1h', min_periods=D // 48).std()
    df['vol_ratio_1h_24h'] = (vol1h / (sigma24 + eps)).clip(0, 5).values

    hi24 = lp.rolling('24h', min_periods=D // 4).max()
    lo24 = lp.rolling('24h', min_periods=D // 4).min()
    df['range_pos_24h'] = ((lp - lo24) / (hi24 - lo24 + eps)).clip(0, 1).values

    if all(c in df.columns for c in ['spot_bid_close', 'spot_ask_close']):
        spot_mid = (df['spot_bid_close'] + df['spot_ask_close']) / 2.0
        basis = pd.Series(((df['close_price'] - spot_mid) / (spot_mid + eps)).values,
                          index=df['dt'])
        b_ma = basis.rolling('4h', min_periods=D // 12).mean()
        b_sd = basis.rolling('4h', min_periods=D // 12).std()
        df['basis_z_4h'] = ((basis - b_ma) / (b_sd + eps)).clip(-8, 8).values
        b_mom = basis - past(basis, pd.Timedelta(hours=1))
        b_mom_sd = b_mom.rolling('4h', min_periods=D // 12).std()
        df['basis_mom_1h'] = (b_mom / (b_mom_sd + eps)).clip(-8, 8).values
    else:
        df['basis_z_4h'] = np.nan
        df['basis_mom_1h'] = np.nan

    dow = df['dt'].dt.dayofweek
    df['dow_sin'] = np.sin(2 * np.pi * dow / 7)
    df['dow_cos'] = np.cos(2 * np.pi * dow / 7)

    # ── run.006: schema-3/4 microstructure features ──────────────────────
    # Same conventions as the run.004 block: causal time-based rollers,
    # z-scores clipped ±8, ratios clipped [0, 10]. Every group is guarded on
    # column presence so pre-v3 files simply yield NaN (the era slice and the
    # coverage guard in the next cell handle the rest).
    mp1h = D // 24                                    # 1h of bars = min_periods

    def _ts(x):
        return pd.Series(np.asarray(x, dtype=np.float64), index=df['dt'])

    def roll_z(x, win='4h', mp=mp1h):
        s = _ts(x)
        return ((s - s.rolling(win, min_periods=mp).mean()) /
                (s.rolling(win, min_periods=mp).std() + eps)).clip(-8, 8).values

    def roll_ratio(x, win='4h', mp=mp1h, hi=10.0):
        s = _ts(x)
        return (s / (s.rolling(win, min_periods=mp).mean() + eps)).clip(0, hi).values

    have = set(df.columns)

    df['ofi_z'] = roll_z(df['future_ofi_sum']) if 'future_ofi_sum' in have else np.nan

    flow_near = ['future_add_bid_near_sum', 'future_cancel_bid_near_sum',
                 'future_add_ask_near_sum', 'future_cancel_ask_near_sum']
    if set(flow_near) <= have:
        ab, cb = df[flow_near[0]], df[flow_near[1]]
        aa, ca = df[flow_near[2]], df[flow_near[3]]
        df['flow_net_near_z']   = roll_z((ab - cb) - (aa - ca))
        df['cancel_imbal_near'] = ((cb - ca) / (cb + ca + eps)).values
        # bid pulled faster than replenished = support evaporating (log-diff
        # is scale-free, so no rolling normalisation needed)
        df['pull_add_bid'] = (np.log1p(cb) - np.log1p(ab)).clip(-8, 8)
        df['pull_add_ask'] = (np.log1p(ca) - np.log1p(aa)).clip(-8, 8)
    else:
        for c in ['flow_net_near_z', 'cancel_imbal_near', 'pull_add_bid', 'pull_add_ask']:
            df[c] = np.nan

    flow_wide = ['future_add_bid_wide_sum', 'future_cancel_bid_wide_sum',
                 'future_add_ask_wide_sum', 'future_cancel_ask_wide_sum']
    if set(flow_wide) <= have and set(flow_near) <= have:
        # the wide band INCLUDES the near band by construction — subtract to
        # get the 0.05–0.25% ring only (schema-4 files; NaN in out3 hours)
        abx = (df[flow_wide[0]] - df[flow_near[0]]).clip(lower=0)
        cbx = (df[flow_wide[1]] - df[flow_near[1]]).clip(lower=0)
        aax = (df[flow_wide[2]] - df[flow_near[2]]).clip(lower=0)
        cax = (df[flow_wide[3]] - df[flow_near[3]]).clip(lower=0)
        df['flow_net_widex_z'] = roll_z((abx - cbx) - (aax - cax))
    else:
        df['flow_net_widex_z'] = np.nan

    df['micro_dev_z'] = (roll_z(df['future_micro_dev_close'])
                         if 'future_micro_dev_close' in have else np.nan)
    df['mid_rv_norm']    = roll_ratio(df['future_mid_rv'])    if 'future_mid_rv'    in have else np.nan
    df['mid_flips_norm'] = roll_ratio(df['future_mid_flips']) if 'future_mid_flips' in have else np.nan
    df['msg_rate_norm']  = (roll_ratio(df['future_depth_msg_count'])
                            if 'future_depth_msg_count' in have else np.nan)

    if {'future_bid_depletion_count', 'future_ask_depletion_count'} <= have:
        bd, ad = df['future_bid_depletion_count'], df['future_ask_depletion_count']
        df['depl_imbal']      = ((bd - ad) / (bd + ad + 1.0)).values
        df['depl_total_norm'] = roll_ratio(bd + ad)
    else:
        df['depl_imbal'] = np.nan
        df['depl_total_norm'] = np.nan

    for side in ('buy', 'sell'):
        med, p90 = f'future_{side}_size_median', f'future_{side}_size_p90'
        if {med, p90} <= have:
            # fat right tail at equal volume = informed flow; log1p of the
            # ratio is scale-free across the BTC-qty drift
            df[f'{side}_tail_ratio'] = np.log1p((df[p90] / (df[med] + eps)).clip(0, 100))
        else:
            df[f'{side}_tail_ratio'] = np.nan

    # liquidations: an absent liquidation IS zero, so cnt/notional NaNs are
    # filled with 0 — but only inside the schema-3+ era, so the coverage
    # guard still sees pre-v3 rows as missing.
    m3 = df['future_ofi_sum'].notna() if 'future_ofi_sum' in have else pd.Series(False, index=df.index)
    liq_cols = ['opt_long_force_exit_cnt_sum', 'opt_short_force_exit_cnt_sum',
                'opt_long_force_exit_notional_sum', 'opt_short_force_exit_notional_sum',
                'opt_force_exit_notional_max']
    for c in liq_cols:
        if c in have:
            df.loc[m3, c] = df.loc[m3, c].fillna(0)
    if {'opt_long_force_exit_cnt_sum', 'opt_short_force_exit_cnt_sum'} <= have:
        df['liq_cnt_log'] = np.log1p(df['opt_long_force_exit_cnt_sum'] +
                                     df['opt_short_force_exit_cnt_sum'])
    else:
        df['liq_cnt_log'] = np.nan
    lqf = df.get('opt_long_force_exit_qty_sum',  pd.Series(np.nan, index=df.index)).fillna(0)
    sqf = df.get('opt_short_force_exit_qty_sum', pd.Series(np.nan, index=df.index)).fillna(0)
    df['liq_imbal'] = ((lqf - sqf) / (lqf + sqf + eps)).values
    if {'opt_long_force_exit_notional_sum', 'opt_short_force_exit_notional_sum'} <= have:
        df['liq_notional_log'] = np.log1p(df['opt_long_force_exit_notional_sum'] +
                                          df['opt_short_force_exit_notional_sum'])
    else:
        df['liq_notional_log'] = np.nan
    df['liq_notional_max_log'] = (np.log1p(df['opt_force_exit_notional_max'])
                                  if 'opt_force_exit_notional_max' in have else np.nan)

    wall_cols = ['future_bid_wall_qty_median', 'future_ask_wall_qty_median',
                 'future_bid_wall_dist_median', 'future_ask_wall_dist_median']
    if set(wall_cols) <= have:
        bq, aq = df[wall_cols[0]], df[wall_cols[1]]
        df['wall_imbal'] = ((bq - aq) / (bq + aq + eps)).values
        # dist −1 = no level in band → map to the band edge ("wall absent" =
        # "as far away as possible"); qty 0 already carries the absence signal
        df['bid_wall_dist'] = df[wall_cols[2]].where(df[wall_cols[2]] >= 0, WALL_BAND)
        df['ask_wall_dist'] = df[wall_cols[3]].where(df[wall_cols[3]] >= 0, WALL_BAND)
        df['wall_qty_norm'] = roll_ratio(bq + aq)
    else:
        for c in ['wall_imbal', 'bid_wall_dist', 'ask_wall_dist', 'wall_qty_norm']:
            df[c] = np.nan

    if {'future_max_batch_buy_qty', 'future_max_buy_run_qty'} <= have:
        # concentration of the bar's volume in one 200ms burst / one same-side
        # sweep run — bounded [0,1] per side, no normalisation needed
        df['batch_buy_rel']  = (df['future_max_batch_buy_qty']  / (df['future_buy_qty']  + eps)).clip(0, 1)
        df['batch_sell_rel'] = (df['future_max_batch_sell_qty'] / (df['future_sell_qty'] + eps)).clip(0, 1)
        df['sweep_buy_rel']  = (df['future_max_buy_run_qty']    / (df['future_buy_qty']  + eps)).clip(0, 1)
        df['sweep_sell_rel'] = (df['future_max_sell_run_qty']   / (df['future_sell_qty'] + eps)).clip(0, 1)
    else:
        for c in ['batch_buy_rel', 'batch_sell_rel', 'sweep_buy_rel', 'sweep_sell_rel']:
            df[c] = np.nan

    if {'opt_eth_mid_open', 'opt_eth_mid_close'} <= have:
        eo, ec = df['opt_eth_mid_open'], df['opt_eth_mid_close']
        eth_ret = pd.Series(np.where((eo > 0) & (ec > 0), np.log(ec / eo), np.nan),
                            index=df['dt'])
        df['eth_ret_z'] = (eth_ret / (eth_ret.rolling('4h', min_periods=mp1h).std() + eps)
                           ).clip(-8, 8).values
        # ETH ran but BTC has not (yet): 1-min cumulative z-gap
        er4 = eth_ret.rolling(4, min_periods=4).sum()
        br4 = r1.rolling(4, min_periods=4).sum()
        er4z = er4 / (er4.rolling('4h', min_periods=mp1h).std() + eps)
        br4z = br4 / (br4.rolling('4h', min_periods=mp1h).std() + eps)
        df['eth_lead_gap'] = (er4z - br4z).clip(-8, 8).values
    else:
        df['eth_ret_z'] = np.nan
        df['eth_lead_gap'] = np.nan

    df['lsr_z'] = (roll_z(df['opt_long_short_ratio_sample'], '24h', D // 4)
                   if 'opt_long_short_ratio_sample' in have else np.nan)

    # Scattered-NaN guard: a sample needs 64 contiguous surviving rows behind
    # it and 240 ahead, so a feature with even ~2% RANDOMLY scattered NaNs
    # (an ETH bar with no data, a missing long/short-ratio sample, a 4-bar
    # rolling gap echo) would wipe out most valid samples via dropna. These
    # per-bar signals are neutral-at-0 by construction (z-scores / gaps), so
    # fill them with 0 inside the schema-3+ era; warmup and pre-era rows stay
    # NaN so the coverage guard still sees them.
    for c in ['eth_ret_z', 'eth_lead_gap', 'lsr_z']:
        bad = m3 & df[c].isna() & (df['dt'] > df.loc[m3, 'dt'].iloc[0] +
                                   pd.Timedelta('24h')) if m3.any() else None
        if bad is not None:
            df.loc[bad, c] = 0.0

    del lp, r1, sigma24, vol1h, hi24, lo24
    # ──────────────────────────────────────────────────────────────────────

    # Targets: volatility-normalised future return per horizon. A target is
    # valid only when the row h bars ahead really is h*15s ahead — no labels
    # across collector gaps. vol uses only past prices (no leakage).
    # (datetime64[s] cast is robust to the pandas 2/3 resolution change)
    dt_sec = pd.Series(df['dt'].values.astype('datetime64[s]').astype(np.int64),
                       index=df.index)
    for h in horizons:
        vol   = df['close_price'].diff(h).rolling(vol_window, min_periods=vol_window // 4).std()
        delta = df['close_price'].shift(-h) - df['close_price']
        y     = (delta / (vol + eps)).clip(-TARGET_CLIP, TARGET_CLIP)
        valid = (dt_sec.shift(-h) - dt_sec) == h * WINDOW_SEC
        y[~valid.fillna(False)] = np.nan
        df[f'y_{h}'] = y

    # ── run.006/007: tradable-opportunity labels ─────────────────────────
    # lup_h = 1 if the close path within the next h bars rises ≥ THETA_BY_H[h]
    # above the current close (max favourable excursion for a long entered
    # now); ldn_h mirrors it for a short. θ is per-horizon in run.007 (15 bps
    # for h ≤ 3 where 20 bps is near-unreachable, 20 bps = 2× taker RT
    # otherwise). Same validity rule as the regression targets: bars can
    # never be closer than 15s, so an exact h·15s span to row i+h implies
    # every intermediate bar is present.
    cvals = df['close_price'].values.astype(np.float64)
    n_rows = len(cvals)
    for h in horizons:
        fmax = np.full(n_rows, np.nan)
        fmin = np.full(n_rows, np.nan)
        if n_rows > h:
            wv = np.lib.stride_tricks.sliding_window_view(cvals, h)  # wv[j] = cvals[j:j+h]
            fmax[:n_rows - h] = wv[1:].max(axis=1)   # windows starting at i+1
            fmin[:n_rows - h] = wv[1:].min(axis=1)
        valid = ((dt_sec.shift(-h) - dt_sec) == h * WINDOW_SEC).fillna(False).values
        th = THETA_BY_H[h]
        up = (fmax / cvals - 1.0) >= th
        dn = (1.0 - fmin / cvals) >= th
        df[f'lup_{h}'] = np.where(valid, up.astype(np.float32), np.nan)
        df[f'ldn_{h}'] = np.where(valid, dn.astype(np.float32), np.nan)
    return df

print('Feature pipeline loaded.')
print(f'Binary heads: {CLS_COLS}')


In [ ]:
# ── Cell 5: Load data, era slice, coverage guard, build arrays ────────────
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')) +
               glob.glob(os.path.join(DATA_DIR, '*.csv.gz')))
print(f'Found {len(files)} files')

raw = load_files(files)
print(f'Total rows loaded: {len(raw):,}')

df = prepare(raw, HORIZONS, VOL_WINDOW)
del raw; gc.collect()

# ── run.006: restrict the analysis window to the schema-3+ era ───────────
# The v3 features only exist where the v3/v4 collector ran; over the full
# ~13-month range they would all fail the coverage guard. Features are
# computed on the FULL frame first (so the 24h rollers are warm at the era
# boundary), then the frame is sliced.
if V3_ERA_ONLY:
    mark = (df[V3_MARKER].notna() if V3_MARKER in df.columns
            else pd.Series(False, index=df.index))
    assert mark.any(), (
        f'No schema-3 data found ({V3_MARKER} entirely missing). Either the '
        f'out3/out4 files are not in btc_data.tar.xz — rebuild the tar — or '
        f'set V3_ERA_ONLY = False to run on v1 features only.')
    v3_start = df.loc[mark, 'dt'].iloc[0]
    era_days = (df['dt'].iloc[-1] - v3_start).total_seconds() / 86400
    n_pre = int((df['dt'] < v3_start).sum())
    print(f'\nSchema-3+ era starts {v3_start}  ({era_days:.1f} days, '
          f'dropping {n_pre:,} pre-era rows)')
    assert era_days >= MIN_ERA_DAYS, (
        f'Only {era_days:.1f} days of schema-3+ data (< {MIN_ERA_DAYS}). '
        f'Let the collector accumulate more before running run.006.')
    df = df[df['dt'] >= v3_start].reset_index(drop=True)
    if 'schema_version' in df.columns:
        sv = df['schema_version']
        print('  schema mix: ' + '  '.join(
            f'v{int(v)}: {(sv == v).mean()*100:.1f}%' for v in sorted(sv.dropna().unique())))

# Coverage guard (within the analysis window): schema-4-only columns (wide
# band, walls, bursts, ETH) are NaN in out3 hours; anything below
# COVERAGE_MIN is excluded rather than letting dropna() eat the window.
features = []
for f_ in ALL_FEATURES:
    if f_ not in df.columns:
        continue
    cov = df[f_].notna().mean()
    if cov >= COVERAGE_MIN:
        features.append(f_)
    else:
        print(f'  EXCLUDED {f_:22s} coverage {cov*100:5.1f}% < {COVERAGE_MIN*100:.0f}%')
kept_v3 = [f_ for f_ in features if f_ in V3_FEATURES]
print(f'Features kept: {len(features)}/{len(ALL_FEATURES)} '
      f'(schema-3/4: {len(kept_v3)}/{len(V3_FEATURES)})')
# run.006 is built to test the schema-3/4 features, but it degrades to the
# v1 feature subset when those columns are absent (v1 collector files, or a
# btc_data.tar.xz that dropped the out3/out4 files) instead of aborting.
V3_AVAILABLE = len(kept_v3) >= 10
if not V3_AVAILABLE:
    print(f'\n*** v1-compat mode: only {len(kept_v3)}/{len(V3_FEATURES)} schema-3/4 '
          f'features present ***')
    print('    The loaded files predate the schema-3/4 collector (no future_ofi_sum, '
          'walls, ETH lead-lag, ...),')
    print('    so the selective/opportunity heads train on the v1 feature subset only. '
          'This does NOT')
    print('    test run.006\'s schema-3/4 hypothesis — it reruns the run.004/005 feature '
          'set under the')
    print('    selective objective. Feed out3/out4 files (rebuild btc_data.tar.xz) for '
          'the intended run.')

n0 = len(df)
clean = df[features + [f'y_{h}' for h in HORIZONS] + CLS_COLS + ['dt', 'close_price']] \
          .dropna(subset=features).reset_index(drop=True)
lost = n0 - len(clean)
print(f'Rows: {n0:,} → {len(clean):,} after dropna ({lost/max(n0,1)*100:.2f}% dropped)')
if lost > 0.05 * n0:
    print('\n*** WARNING: dropna removed >5% of rows — check per-feature NaN fractions: ***')
    print(df[features].isna().mean().sort_values(ascending=False).head(10))
del df; gc.collect()

X_raw  = clean[features].values.astype(np.float32)
Y_all  = np.stack([clean[f'y_{h}'].values for h in HORIZONS], axis=1).astype(np.float32)
L_all  = np.stack([clean[c].values for c in CLS_COLS], axis=1).astype(np.float32)
dt_s   = clean['dt'].values.astype('datetime64[s]').astype(np.int64)
dt_all = clean['dt'].values
close  = clean['close_price'].values.astype(np.float64)
n, F_DIM = X_raw.shape
del clean; gc.collect()

# A sample "ends" at row i: input window = rows [i-SEQ_LEN+1, i], targets and
# labels at/through i+h. Backward validity: the window must be contiguous 15s
# bars. Forward validity: row i+MAX_H must be exactly MAX_H·15s ahead IN THIS
# (post-dropna) array — dropna can remove interior rows, and the evaluation
# below indexes close[e + h] positionally, so the check must hold here, not
# just in the pre-drop frame. (Improvement over run.005, which relied on the
# pre-drop check only.)
MAX_H = max(HORIZONS)
contig = np.zeros(n, dtype=bool)
span = (SEQ_LEN - 1) * WINDOW_SEC
contig[SEQ_LEN-1:] = (dt_s[SEQ_LEN-1:] - dt_s[:n-SEQ_LEN+1]) == span
fwd_ok = np.zeros(n, dtype=bool)
fwd_ok[:n-MAX_H] = (dt_s[MAX_H:] - dt_s[:n-MAX_H]) == MAX_H * WINDOW_SEC
sample_valid = (contig & fwd_ok &
                np.isfinite(Y_all).all(axis=1) & np.isfinite(L_all).all(axis=1))
assert sample_valid.sum() > 0, 'No valid samples — check bar cadence / targets'

print(f'\nRows: {n:,}   valid samples: {sample_valid.sum():,} ({sample_valid.mean()*100:.1f}%)')
print(f'Date range: {pd.Timestamp(dt_all[0])} → {pd.Timestamp(dt_all[-1])}')
H_TRADE_IDX = HORIZONS.index(H_TRADE)
yt = Y_all[sample_valid, H_TRADE_IDX]
print(f'Trading target y_{H_TRADE}: std={yt.std():.3f}   |y|>1σ: {(np.abs(yt) > 1).mean()*100:.1f}%   '
      f'up share: {(yt > 0).mean()*100:.1f}%')
print('\nOpportunity-label base rates (per-horizon θ, valid samples):')
for j, (h, s) in enumerate(CLS_KEYS):
    print(f'  {CLS_COLS[j]:8s} θ={THETA_BY_H[h]*1e4:2.0f}bp  '
          f'{L_all[sample_valid, j].mean()*100:6.2f}%')


In [ ]:
# ── Cell 6: Walk-forward folds (expanding window, purged) ─────────────────
def ends_in(a, b):
    return np.flatnonzero(sample_valid[a:b]) + a

test_start = int(n * TEST_START_FRAC)
chunk = (n - test_start) // N_FOLDS
folds = []
for k in range(N_FOLDS):
    lo = test_start + k * chunk
    hi = n if k == N_FOLDS - 1 else lo + chunk
    boundary = lo - MAX_H                     # purge: val labels must not reach into test
    val_lo   = int(boundary * (1 - VAL_FRAC))
    folds.append({'k': k,
                  'train_hi': val_lo - MAX_H,  # purge: train labels must not reach into val
                  'val':  (val_lo, boundary),
                  'test': (lo, hi)})

print('Fold   n_train     n_val    n_test    test period')
for f_ in folds:
    t_lo, t_hi = f_['test']
    n_tr = len(ends_in(0, f_['train_hi']))
    n_va = len(ends_in(*f_['val']))
    n_te = len(ends_in(t_lo, t_hi))
    assert min(n_tr, n_va, n_te) > 0, f"fold {f_['k']} has an empty split"
    span_d = (dt_s[t_hi-1] - dt_s[t_lo]) / 86400
    warn = '   *** <7 days — treat this fold as indicative only ***' if span_d < 7 else ''
    print(f"  {f_['k']}  {n_tr:>9,}  {n_va:>8,}  {n_te:>8,}    "
          f"{pd.Timestamp(dt_all[t_lo])} → {pd.Timestamp(dt_all[t_hi-1])} "
          f"({span_d:.1f}d){warn}")


In [ ]:
# ── Cell 7: Model — frozen trunk, regression heads + opportunity heads ────
class BtcLSTMSel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout, n_reg, n_cls):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.reg_head = nn.Linear(hidden_dim, n_reg)
        self.cls_head = nn.Linear(hidden_dim, n_cls)

    def forward(self, x):
        h, _ = self.lstm(self.input_norm(x))
        z = self.drop(h[:, -1])
        return self.reg_head(z), self.cls_head(z)

_tmp = BtcLSTMSel(F_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT, len(HORIZONS), len(CLS_KEYS))
print(f'Model parameters: {sum(p.numel() for p in _tmp.parameters()):,}')
del _tmp


In [ ]:
# ── Cell 8: Training helpers (zero-copy window gathering, multi-seed) ─────
AMP = device.type == 'cuda'

def make_scheduler(opt, warmup, total):
    def fn(ep):
        if ep < warmup:
            return (ep + 1) / warmup
        prog = (ep - warmup) / max(total - warmup, 1)
        return 0.5 * (1 + math.cos(math.pi * prog))
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

def ic(a, b):
    v = spearmanr(a, b)[0]
    return float(v) if np.isfinite(v) else 0.0

def ap(y_true, p):
    # average precision with degenerate-label guards (tiny val slices can
    # have all-0 or all-1 labels; AP is undefined there)
    y = y_true > 0.5
    if y.sum() == 0 or y.all():
        return float(y.mean())
    return float(average_precision_score(y, p))

def predict(model, view, ends, batch=4096):
    model.eval()
    regs, probs = [], []
    with torch.no_grad():
        for s in range(0, len(ends), batch):
            e = ends[s:s+batch]
            xb = torch.from_numpy(view[e - (SEQ_LEN - 1)]).to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=AMP):
                r, lg = model(xb)
            regs.append(r.float().cpu())
            probs.append(torch.sigmoid(lg.float()).cpu())
    return torch.cat(regs).numpy(), torch.cat(probs).numpy()

def train_one_seed(seed, view, train_ends, val_ends, samp_p, pos_w):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

    model = BtcLSTMSel(F_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT,
                       len(HORIZONS), len(CLS_KEYS)).to(device)
    pw = torch.tensor(pos_w, dtype=torch.float32, device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = make_scheduler(opt, WARMUP, EPOCHS)
    scaler_amp = torch.amp.GradScaler('cuda', enabled=AMP)

    best_ap, best_state, best_epoch, wait = -9.0, None, 0, 0
    hist = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        # weighted draw WITH replacement (weighted w/o replacement is O(k·n)
        # in numpy); duplicates are just oversampling, which is the point
        ep_ends = rng.choice(train_ends,
                             min(len(train_ends), EPOCH_SAMPLE_CAP),
                             replace=True, p=samp_p)
        tot, m = 0.0, 0
        for s in range(0, len(ep_ends), BATCH_SIZE):
            e = ep_ends[s:s+BATCH_SIZE]
            xb = torch.from_numpy(view[e - (SEQ_LEN - 1)]).to(device, non_blocking=True)
            yb = torch.from_numpy(Y_all[e]).to(device, non_blocking=True)
            lb = torch.from_numpy(L_all[e]).to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=AMP):
                r, lg = model(xb)
                # regression part: frozen form from run.003/004
                per = F.smooth_l1_loss(r, yb, reduction='none').mean(dim=1)
                w = torch.clamp(1.0 + LOSS_W_ALPHA * yb[:, H_TRADE_IDX].abs(),
                                max=LOSS_W_CAP)
                huber = (w * per).sum() / w.sum()
                # run.006: opportunity heads (autocast runs BCE in fp32)
                bce = F.binary_cross_entropy_with_logits(lg, lb, pos_weight=pw)
                loss = huber + BCE_W * bce
            opt.zero_grad(set_to_none=True)
            scaler_amp.scale(loss).backward()
            scaler_amp.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler_amp.step(opt)
            scaler_amp.update()
            tot += loss.item() * len(e); m += len(e)
        sched.step()

        vr, vp = predict(model, view, val_ends)
        v_ic = [ic(vr[:, i], Y_all[val_ends, i]) for i in range(len(HORIZONS))]
        v_ap_up = ap(L_all[val_ends, SEL_UP_J], vp[:, SEL_UP_J])
        v_ap_dn = ap(L_all[val_ends, SEL_DN_J], vp[:, SEL_DN_J])
        cur = 0.5 * (v_ap_up + v_ap_dn)      # early-stop metric: the run's object
        hist.append({'train_loss': tot / m, 'val_ic': v_ic,
                     'val_ap_up': v_ap_up, 'val_ap_dn': v_ap_dn})
        if cur > best_ap:
            best_ap, best_epoch, wait = cur, epoch, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            flag = '  ← best'
        else:
            wait += 1
            flag = f'  ({wait}/{PATIENCE})'
        print(f'    seed {seed}  ep {epoch:2d}  loss={tot/m:.4f}  '
              f'IC_h{H_SEL}={v_ic[H_TRADE_IDX]:+.4f}  '
              f'AP_up{H_SEL}={v_ap_up:.4f}  AP_dn{H_SEL}={v_ap_dn:.4f}{flag}')
        if wait >= PATIENCE:
            print('    early stop')
            break

    model.load_state_dict(best_state)
    return model, best_state, best_epoch, hist

def train_fold(fold):
    t_hi = fold['train_hi']
    v0, v1 = fold['val']
    s0, s1 = fold['test']

    scaler = StandardScaler().fit(X_raw[:t_hi])
    Xs = scaler.transform(X_raw).astype(np.float32)
    # (n-SEQ_LEN+1, SEQ_LEN, F) VIEW into Xs — no copy; batches materialise
    # only when fancy-indexed below. This is what keeps RAM under control.
    view = np.lib.stride_tricks.sliding_window_view(Xs, (SEQ_LEN, F_DIM))[:, 0]

    train_ends = ends_in(0, t_hi)
    val_ends   = ends_in(v0, v1)
    test_ends  = ends_in(s0, s1)

    # Importance sampling: bars with bigger vol-normalised moves are drawn
    # more often; SAMPLE_W_BASE keeps flat bars in the mix so the model still
    # learns what "nothing happening" looks like.
    samp_w = SAMPLE_W_BASE + np.abs(Y_all[train_ends, H_TRADE_IDX]).astype(np.float64)
    samp_p = samp_w / samp_w.sum()
    eff = 1.0 / np.sum(samp_p ** 2) / len(samp_p)
    print(f'  sampling tilt: effective sample fraction {eff*100:.1f}% '
          f'(100% = uniform)')

    # per-head pos_weight from TRAIN base rates only (no val/test peeking)
    pos = L_all[train_ends].mean(axis=0)
    pos_w = np.clip((1.0 - pos) / np.maximum(pos, 1e-6), 1.0, POS_W_CAP)
    print('  train base rates: ' + '  '.join(
        f'{c}={p*100:.1f}%(pw{w:.1f})' for c, p, w in zip(CLS_COLS, pos, pos_w)))

    states, best_epochs, hists = [], [], []
    val_preds, test_preds, seed_test_ic = [], [], []
    seed_val_probs, seed_test_probs = [], []
    for seed in SEEDS:
        model, best_state, best_epoch, hist = train_one_seed(
            seed, view, train_ends, val_ends, samp_p, pos_w)
        vr, vp = predict(model, view, val_ends)
        tr, tp = predict(model, view, test_ends)
        s_ic = [ic(tr[:, i], Y_all[test_ends, i]) for i in range(len(HORIZONS))]
        s_ap = [ap(L_all[test_ends, j], tp[:, j]) for j in (SEL_UP_J, SEL_DN_J)]
        print(f'    seed {seed} TEST: IC_h{H_SEL}={s_ic[H_TRADE_IDX]:+.4f}  '
              f'AP_up{H_SEL}={s_ap[0]:.4f}  AP_dn{H_SEL}={s_ap[1]:.4f}   '
              f'(best epoch {best_epoch})')
        states.append(best_state); best_epochs.append(best_epoch); hists.append(hist)
        val_preds.append(vr); test_preds.append(tr); seed_test_ic.append(s_ic)
        seed_val_probs.append(vp); seed_test_probs.append(tp)
        del model

    val_pred  = np.mean(val_preds, axis=0)          # seed-ensemble (regression)
    test_pred = np.mean(test_preds, axis=0)
    val_prob  = np.mean(seed_val_probs, axis=0)     # seed-ensemble (probability)
    test_prob = np.mean(seed_test_probs, axis=0)
    test_ic = [ic(test_pred[:, i], Y_all[test_ends, i]) for i in range(len(HORIZONS))]
    test_ap = [ap(L_all[test_ends, j], test_prob[:, j]) for j in range(len(CLS_KEYS))]
    out = {'scaler': scaler, 'states': states, 'best_epochs': best_epochs,
           'hist': hists[-1], 'val_ends': val_ends, 'val_pred': val_pred,
           'test_ends': test_ends, 'test_pred': test_pred,
           'test_ic': test_ic, 'seed_test_ic': seed_test_ic,
           'val_prob': val_prob, 'test_prob': test_prob,
           'seed_val_probs': seed_val_probs, 'seed_test_probs': seed_test_probs,
           'test_ap': test_ap}
    del Xs, view
    gc.collect()
    return out


In [ ]:
# ── Cell 9: Run walk-forward ──────────────────────────────────────────────
results = []
for fold in folds:
    lo, hi = fold['test']
    print(f"\n════ Fold {fold['k']}: test {pd.Timestamp(dt_all[lo])} → {pd.Timestamp(dt_all[hi-1])} ════")
    res = train_fold(fold)
    results.append(res)
    ics = '  '.join(f'IC_h{h}={v:+.4f}' for h, v in zip(HORIZONS, res['test_ic']))
    print(f"  ENSEMBLE TEST ({len(res['test_ends']):,} samples): {ics}")

print('\n──── Regression reference (seed-ensemble; ± = per-seed spread) ────')
print('NOTE: this window differs from runs 003–005, so these ICs are context,')
print('not comparable numbers. run.004 per-seed noise floor was ±0.002–0.018.')
hdr = '  '.join(f"{'IC_h' + str(h):>16s}" for h in HORIZONS)
print(f'Fold    n_test   {hdr}')
for f_, r in zip(folds, results):
    cells = []
    for j in range(len(HORIZONS)):
        s_ics = [r['seed_test_ic'][s][j] for s in range(len(SEEDS))]
        cells.append(f"{r['test_ic'][j]:>+8.4f} ±{np.std(s_ics):.4f}")
    print(f"  {f_['k']}   {len(r['test_ends']):>8,}  " + '  '.join(cells))

ends_pool = np.concatenate([r['test_ends'] for r in results])
pred_pool = np.concatenate([r['test_pred'] for r in results])
prob_pool = np.concatenate([r['test_prob'] for r in results])
pooled_ic = [ic(pred_pool[:, i], Y_all[ends_pool, i]) for i in range(len(HORIZONS))]
print('Pooled ensemble:   ' + '  '.join(f'{v:>+9.4f}       ' for v in pooled_ic))

print('\n──── Opportunity-head test AP (seed-ensemble) vs base rate ────')
print('AP ≈ base rate = the head ranks no better than chance; the selective')
print('tables in the next cell are what actually matters — AP is a whole-')
print('distribution summary, and this run only cares about the extreme tail.')
print(f'{"head":8s}  ' + '  '.join(f'fold {k}' for k in range(N_FOLDS)) + '   pooled  base%')
for j, c in enumerate(CLS_COLS):
    aps = '  '.join(f"{r['test_ap'][j]:.4f}" for r in results)
    pooled_ap = ap(L_all[ends_pool, j], prob_pool[:, j])
    base = L_all[ends_pool, j].mean()
    print(f'{c:8s}  {aps}   {pooled_ap:.4f}  {base*100:5.2f}')

# Backtest/pooling score: per-fold predictions divided by that fold's
# VALIDATION score std, so "σ" thresholds mean the same thing in every fold
# and the score scale is chosen without touching test data.
score_pool = np.concatenate([r['test_pred'] / (r['val_pred'].std(axis=0) + 1e-9)
                             for r in results])

def daily_ic_stats(ends, score, y):
    days = dt_all[ends].astype('datetime64[D]')
    vals = []
    for d in np.unique(days):
        m = days == d
        if m.sum() >= MIN_DAY_SAMPLES:
            vals.append(ic(score[m], y[m]))
    vals = np.array(vals)
    se = vals.std(ddof=1) / np.sqrt(len(vals))
    return vals.mean(), se, vals.mean() / se, len(vals)

print('\n──── Daily-IC t-stats (pooled ensemble; |t|>2 ≈ significant) ────')
for j, h in enumerate(HORIZONS):
    mu, se, t, k = daily_ic_stats(ends_pool, score_pool[:, j], Y_all[ends_pool, j])
    print(f'  h{h:<4d}  mean daily IC {mu:+.4f} ± {se:.4f}   t = {t:+.2f}   ({k} days)')

# ── persist scores IMMEDIATELY — run.004's scores died with the VM ────────
# (rolling-threshold evaluation is fully reproducible offline from this npz:
#  it saves val probs + val/test timestamps per fold)
fold_of = np.concatenate([np.full(len(r['test_ends']), f_['k'])
                          for f_, r in zip(folds, results)])
val_ends_pool = np.concatenate([r['val_ends'] for r in results])
val_prob_pool = np.concatenate([r['val_prob'] for r in results])
val_fold_of   = np.concatenate([np.full(len(r['val_ends']), f_['k'])
                                for f_, r in zip(folds, results)])
seed_prob_pool     = np.concatenate([np.stack(r['seed_test_probs'], axis=2) for r in results])
seed_val_prob_pool = np.concatenate([np.stack(r['seed_val_probs'],  axis=2) for r in results])
scores_path = os.path.join(OUTPUT_DIR, 'run008_scores.npz')
np.savez_compressed(scores_path,
    ends=ends_pool, score=score_pool, prob=prob_pool, seed_prob=seed_prob_pool,
    labels=L_all[ends_pool], y=Y_all[ends_pool],
    horizons=np.array(HORIZONS), cls_cols=np.array(CLS_COLS),
    dt_s=dt_s[ends_pool], close=close[ends_pool], fold_of=fold_of,
    val_ends=val_ends_pool, val_prob=val_prob_pool, val_fold_of=val_fold_of,
    seed_val_prob=seed_val_prob_pool, val_dt_s=dt_s[val_ends_pool])
print(f'\nScores persisted → {scores_path}')
print('(copied to Drive in the final cell — if you plan to stop early, copy it now)')


In [ ]:
# ── Cell 10: Selective evaluation — rolling-causal triggers + TP/SL exits ──
# Two pre-registered changes vs run.006:
#
# 1. TRIGGER RULE ('ens'/'unan'/'reg' variants): τ is recomputed once per test
#    DAY as the (1−rate)-quantile of the scores seen in the trailing
#    ROLL_CAL_DAYS window — val scores seed it, already-seen test scores join
#    it. Deployable (past data only) and immune to run.006's fold-1 pathology
#    (frozen val τ above every test prob → an entire fold with 0 triggers).
#    The frozen-val rule of run.006 is still printed ('val' variant).
#
# 2. EXIT: run.006 scored hold-to-horizon endpoints and the θ-touch had
#    reverted by bar 40 ("lift good but net ≤ 0" — its diagnosed failure).
#    Every trigger now also runs through a TP/SL simulator:
#      entry  taker at close[e]  (signal-bar close, zero latency assumed)
#      TP     limit at entry·(1 + sgn·tp): counted filled MAKER at the limit
#             price on the first bar whose CLOSE trades through it (fill then
#             guaranteed; queue position ignored)
#      SL     first close beyond −sl → exit TAKER at that bar's actual CLOSE
#             (gap-through conservative, not at the stop price)
#      else   time-stop TAKER at close[e+h]
#    Closes of the mid only: intra-bar SL-then-TP sequences are invisible and
#    intra-bar touches that closes miss are lost. No slippage model.
#
# hit%  = path-label rate among triggers (TP-order upper bound)
# sim   = TP/SL-simulated net bps  ← headline; the CI column belongs to it
# endpt = hold-to-horizon endpoint net bps (run.006's lower bound)

RT_BPS    = 2 * TAKER_FEE * 1e4              # taker in + taker out
TP_RT_BPS = (TAKER_FEE + MAKER_FEE) * 1e4    # taker in + maker TP out
DAY_S     = 86400

def day_of(e):
    return dt_all[e].astype('datetime64[D]')

def net_bps(e, side, h):
    fwd = close[e + h] / close[e] - 1.0
    sgn = 1.0 if side == 'up' else -1.0
    return sgn * fwd * 1e4 - RT_BPS

def sim_exit(e, side, h, tp, sl):
    # → (net bps per trade, exit offset in bars 1..h). Vectorised over triggers;
    # close[e+1 .. e+h] is guaranteed contiguous by the sample-validity mask.
    if len(e) == 0:
        return np.zeros(0), np.zeros(0, dtype=int)
    sgn  = 1.0 if side == 'up' else -1.0
    path = close[e[:, None] + np.arange(1, h + 1)[None, :]]
    ret  = sgn * (path / close[e][:, None] - 1.0)
    big  = h + 1
    hit_tp = ret >= tp
    k_tp = np.where(hit_tp.any(axis=1), hit_tp.argmax(axis=1), big)
    if np.isfinite(sl):
        hit_sl = ret <= -sl
        k_sl = np.where(hit_sl.any(axis=1), hit_sl.argmax(axis=1), big)
    else:
        k_sl = np.full(len(e), big)
    net    = ret[:, -1] * 1e4 - RT_BPS                 # default: time stop
    k_exit = np.full(len(e), h)
    rows = np.arange(len(e))
    m = k_sl < k_tp                                    # stop hit first
    net[m]    = ret[rows[m], k_sl[m]] * 1e4 - RT_BPS
    k_exit[m] = k_sl[m] + 1
    m = k_tp < k_sl                                    # TP hit first (a single
    net[m]    = tp * 1e4 - TP_RT_BPS                   # close cannot breach both)
    k_exit[m] = k_tp[m] + 1
    return net, k_exit

def rolling_triggers(cal_x, cal_t, test_x, test_t, rate, side='hi'):
    # τ per test day from the trailing ROLL_CAL_DAYS of scores (cal = val
    # seeds; test scores join as their day passes). side='lo' thresholds the
    # left tail (regression dn baseline).
    xs = np.concatenate([cal_x, test_x])
    ts = np.concatenate([cal_t, test_t])
    o = np.argsort(ts, kind='stable')
    xs, ts = xs[o], ts[o]
    q = 1 - rate if side == 'hi' else rate
    trig = np.zeros(len(test_x), dtype=bool)
    days = test_t // DAY_S
    for d in np.unique(days):
        lo = np.searchsorted(ts, (d - ROLL_CAL_DAYS) * DAY_S)
        hi = np.searchsorted(ts, d * DAY_S)
        if hi - lo < MIN_CAL_N:
            lo = 0                       # thin window → all past scores
        if hi - lo == 0:
            continue                     # nothing causal available: abstain
        tau = np.quantile(xs[lo:hi], q)
        m = days == d
        trig[m] = (test_x[m] >= tau) if side == 'hi' else (test_x[m] <= tau)
    return trig

def fold_triggers(r, j, rate, variant):
    h, side = CLS_KEYS[j]
    vt, tt = dt_s[r['val_ends']], dt_s[r['test_ends']]
    if variant == 'reg':                 # frozen-regression baseline, rolling τ
        ridx = HORIZONS.index(h)
        return rolling_triggers(r['val_pred'][:, ridx], vt,
                                r['test_pred'][:, ridx], tt, rate,
                                'hi' if side == 'up' else 'lo')
    if variant == 'val':                 # run.006's frozen-val rule (comparison)
        tau = np.quantile(r['val_prob'][:, j], 1 - rate)
        return r['test_prob'][:, j] >= tau
    trig = rolling_triggers(r['val_prob'][:, j], vt,
                            r['test_prob'][:, j], tt, rate)
    if variant == 'unan':                # all seeds beyond their own rolling τ
        for s in range(len(SEEDS)):
            trig &= rolling_triggers(r['seed_val_probs'][s][:, j], vt,
                                     r['seed_test_probs'][s][:, j], tt, rate)
    return trig

def boot_ci_mean(x, days, n_boot=N_BOOT, seed=0):
    if len(x) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    uds, inv = np.unique(days, return_inverse=True)
    sums = np.bincount(inv, weights=x, minlength=len(uds))
    cnts = np.bincount(inv, minlength=len(uds)).astype(np.float64)
    idx = rng.integers(0, len(uds), size=(n_boot, len(uds)))
    bs = sums[idx].sum(axis=1) / np.maximum(cnts[idx].sum(axis=1), 1.0)
    return (float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5)))

def trigger_stats(j, rate, variant='ens', tp_mult=TP_MULT, sl_mult=SL_MULT):
    h, side = CLS_KEYS[j]
    th = THETA_BY_H[h]
    per_fold, E, NB, SN, HT = [], [], [], [], []
    for f_, r in zip(folds, results):
        trig = fold_triggers(r, j, rate, variant)
        e = r['test_ends'][trig]
        base = float(L_all[r['test_ends'], j].mean())
        nb = net_bps(e, side, h)
        sn, _ = sim_exit(e, side, h, tp_mult * th, sl_mult * th)
        n_days = max(len(np.unique(day_of(r['test_ends']))), 1)
        hit = float(L_all[e, j].mean()) if len(e) else np.nan
        per_fold.append(dict(k=f_['k'], n=len(e), per_day=len(e) / n_days,
                             hit=hit, base=base,
                             lift=(hit / base) if (len(e) and base > 0) else np.nan,
                             net=float(nb.mean()) if len(e) else np.nan,
                             sim=float(sn.mean()) if len(e) else np.nan))
        E.append(e); NB.append(nb); SN.append(sn); HT.append(L_all[e, j])
    e = np.concatenate(E); nb = np.concatenate(NB)
    sn = np.concatenate(SN); ht = np.concatenate(HT)
    base_p = float(L_all[ends_pool, j].mean())
    n_days = max(len(np.unique(day_of(ends_pool))), 1)
    hit_p = float(ht.mean()) if len(e) else np.nan
    pooled = dict(n=len(e), per_day=len(e) / n_days, hit=hit_p, base=base_p,
                  lift=(hit_p / base_p) if (len(e) and base_p > 0) else np.nan,
                  net=float(nb.mean()) if len(e) else np.nan,
                  sim=float(sn.mean()) if len(e) else np.nan,
                  ci=boot_ci_mean(sn, day_of(e)))     # CI on the SIMULATED net
    return per_fold, pooled

def _fmt(v, spec, na='     —'):
    return format(v, spec) if np.isfinite(v) else na

def show(j, rate, variant='ens', with_folds=False):
    pf, p = trigger_stats(j, rate, variant)
    h, side = CLS_KEYS[j]
    print(f'  {side}{h:<3d} {rate*100:7.3f}%  {variant:4s}  {p["n"]:>6,}  '
          f'{p["per_day"]:>6.2f}  '
          f'{_fmt(p["hit"]*100 if np.isfinite(p["hit"]) else np.nan, "6.1f")}  '
          f'{p["base"]*100:6.2f}  {_fmt(p["lift"], "5.2f")}  '
          f'{_fmt(p["sim"], "+8.2f")}  [{_fmt(p["ci"][0], "+7.2f")}, {_fmt(p["ci"][1], "+7.2f")}]  '
          f'{_fmt(p["net"], "+8.2f")}')
    if with_folds:
        for r_ in pf:
            print(f'        fold {r_["k"]}: n={r_["n"]:<6,} '
                  f'hit={_fmt(r_["hit"]*100 if np.isfinite(r_["hit"]) else np.nan, "5.1f")}% '
                  f'lift={_fmt(r_["lift"], "5.2f")} '
                  f'sim={_fmt(r_["sim"], "+8.2f")} endpt={_fmt(r_["net"], "+8.2f")} bps')
    return pf, p

HDR = ('  head    rate     var       n    /day    hit%   base%   lift   '
       'sim bps  [95% CI, day-boot]  endpt bps')

print(f'════ h{H_SEL} opportunity heads (θ={THETA_BY_H[H_SEL]*1e4:.0f}bp) — '
      f'the run.007 headline ════')
print(HDR)
for j in (SEL_UP_J, SEL_DN_J):
    for rate in TRIGGER_RATES:
        show(j, rate, 'ens', with_folds=(rate == PRIMARY_RATE))

print('\n════ run.006 frozen-val trigger rule (comparison, not gated) ════')
print(HDR)
for j in (SEL_UP_J, SEL_DN_J):
    show(j, PRIMARY_RATE, 'val')

print('\n════ Seed-unanimity abstention (all 3 seeds beyond their own rolling τ) ════')
print('Realized trigger rate < nominal by design — disagreement = "no idea".')
print(HDR)
for j in (SEL_UP_J, SEL_DN_J):
    for rate in TRIGGER_RATES:
        show(j, rate, 'unan')

print('\n════ Baseline: frozen regression score, same rolling-τ rule ════')
print('If the opportunity heads do not beat this, the BCE heads added nothing')
print('over thresholding the run.004-style score.')
print(HDR)
for j in (SEL_UP_J, SEL_DN_J):
    for rate in TRIGGER_RATES:
        show(j, rate, 'reg')

print(f'\n════ Exploratory: all horizons at the primary rate ({PRIMARY_RATE*100:.1f}%) ════')
print('h2/h3 rows are upper bounds (30–45s trades: entry latency is first-order).')
print(HDR)
for j, (h, s) in enumerate(CLS_KEYS):
    if h != H_SEL:
        show(j, PRIMARY_RATE, 'ens')

print(f'\n════ Exploratory: TP/SL grid at the primary head+rate '
      f'(h{H_SEL} @ {PRIMARY_RATE*100:.1f}% ens) ════')
print('Gated cell 11 uses ONLY (TP_MULT, SL_MULT) = '
      f'({TP_MULT}, {SL_MULT}); the rest is hypothesis-generating.')
print('  head    tp·θ   sl·θ        n     sim bps   [95% CI, day-boot]')
for j in (SEL_UP_J, SEL_DN_J):
    h, side = CLS_KEYS[j]
    for tpm, slm in TP_SL_GRID:
        _, p = trigger_stats(j, PRIMARY_RATE, 'ens', tp_mult=tpm, sl_mult=slm)
        sl_txt = f'{slm:4.1f}' if np.isfinite(slm) else 'none'
        print(f'  {side}{h:<4d}  {tpm:4.1f}   {sl_txt}   {p["n"]:>6,}  '
              f'{_fmt(p["sim"], "+9.2f")}  [{_fmt(p["ci"][0], "+7.2f")}, '
              f'{_fmt(p["ci"][1], "+7.2f")}]')

# ── non-overlapping trade simulation at the primary setting ───────────────
# Per-signal stats above count overlapping triggers (bar i and i+1 both firing
# = the same move twice). This greedily takes one trade at a time with the
# TP/SL exits (an early TP exit frees the book sooner), ignores triggers while
# a trade is open — a floor on realizable activity. 1 BTC per trade.
def non_overlap(j, rate, variant='ens'):
    h, side = CLS_KEYS[j]
    th = THETA_BY_H[h]
    ends = np.concatenate([r['test_ends'][fold_triggers(r, j, rate, variant)]
                           for r in results])   # test ranges ordered, disjoint
    nets, kex = sim_exit(ends, side, h, TP_MULT * th, SL_MULT * th)
    usd, last_exit = [], -1
    for i, e in enumerate(ends):
        if e < last_exit:
            continue
        usd.append(close[e] * nets[i] / 1e4)
        last_exit = e + kex[i]
    usd = np.array(usd)
    tag = f'{side}{h} @{rate*100:.1f}% {variant}'
    if len(usd) == 0:
        print(f'  {tag:24s}  no trades')
        return
    print(f'  {tag:24s}  {len(usd):>5,} trades   total {usd.sum():+10.0f} $   '
          f'mean {usd.mean():+7.1f} $   win {100*(usd > 0).mean():.1f}%')

print(f'\n════ Non-overlapping 1-BTC trades, TP/SL exits '
      f'(tp={TP_MULT}·θ, sl={SL_MULT}·θ, time-stop h) ════')
for variant in ('ens', 'unan'):
    for j in (SEL_UP_J, SEL_DN_J):
        non_overlap(j, PRIMARY_RATE, variant)


In [ ]:
# ── Cell 10M: maker-entry simulation — the run.008 headline ────────────────
# run.007's gate failed in pre-registered failure mode #2: "lift good, sim ≤ 0,
# TP-grid all ≤ 0 → fees eat the touches; the remaining lever is ENTRY price".
# The arithmetic: taker entry pays θ−7bp on a hit vs ≈−18bp on a non-hit →
# breakeven hit ≈ 58.5%; best achieved was 49.7% (unan up6 @0.01%). Full-maker
# RT (4bp) moves breakeven to ≈49% — within one hit-rate point. This cell
# measures whether that point survives an honest fill model.
#
# Fill model (same conservative close-trade-through convention as the TP fill
# in sim_exit): post a limit at close[e]·(1 − sgn·ENTRY_DELTA·θ) after
# ENTRY_DELAY bars; it fills AT THE LIMIT on the first of the next ENTRY_WAIT
# bars whose CLOSE trades through it; unfilled by then → cancel, NO trade.
# After the fill: TP limit at entry·(1+sgn·tp) maker; SL taker at the breaching
# bar's actual close (the fill bar's own close counts — a close that gapped
# through both limit and stop exits immediately); else time-stop taker at
# close[e+h]. The horizon stays anchored at the TRIGGER bar: waiting for a
# fill consumes move window, and the sim pays that cost.
#
# FILL-RATE-HONEST accounting — the three numbers that matter together:
#   sim bps   mean net over FILLED trades (per-trade economics; CI belongs here)
#   fill%     share of triggers that filled — a resting limit misses exactly
#             the moves that leave immediately (adverse selection)
#   EV/trig   sum(filled net)/n_triggers — per-SIGNAL expectation, unfilled = 0.
#             Compare directly against cell 10's taker "sim bps" column.
#   missed    gross endpoint move of UNFILLED triggers + their hit rate: if
#             hit% (missed) ≫ hit% (filled), the limit filters out the winners.

MK_MK_BPS = 2 * MAKER_FEE * 1e4              # maker in + maker TP out (4 bp)
MK_TK_BPS = (MAKER_FEE + TAKER_FEE) * 1e4    # maker in + taker out    (7 bp)
H2_IDX    = HORIZONS.index(2)

def sim_maker(e, side, h, tp, sl, wait=ENTRY_WAIT, delta=0.0, delay=ENTRY_DELAY):
    # → (net bps [NaN where unfilled], filled mask, exit offset 1..h,
    #    fill offset 1..h [h+1 where unfilled]) — all full trigger length.
    # tp/sl/delta are ABSOLUTE fractions (callers multiply θ). Vectorised;
    # close[e+1..e+h] contiguity is guaranteed by the sample-validity mask.
    n = len(e)
    if n == 0:
        z = np.zeros(0)
        return z, np.zeros(0, dtype=bool), np.zeros(0, dtype=int), np.zeros(0, dtype=int)
    sgn  = 1.0 if side == 'up' else -1.0
    path = close[e[:, None] + np.arange(1, h + 1)[None, :]]
    lim  = close[e] * (1.0 - sgn * delta)
    relf = sgn * (path / lim[:, None] - 1.0)       # signed return vs the limit
    cols = np.arange(1, h + 1)[None, :]
    big  = h + 1
    can  = (relf <= 0.0) & (cols >= delay + 1) & (cols <= min(delay + wait, h))
    k_fill = np.where(can.any(axis=1), can.argmax(axis=1) + 1, big)
    filled = k_fill <= h
    kf = k_fill[:, None]
    tp_ok = (relf >= tp) & (cols > kf)             # fill-bar close ≤ lim < TP,
    k_tp = np.where(tp_ok.any(axis=1), tp_ok.argmax(axis=1) + 1, big)  # so > kf
    if np.isfinite(sl):                            # loses nothing
        sl_ok = (relf <= -sl) & (cols >= kf)       # fill bar CAN stop out
        k_sl = np.where(sl_ok.any(axis=1), sl_ok.argmax(axis=1) + 1, big)
    else:
        k_sl = np.full(n, big)
    net    = relf[:, -1] * 1e4 - MK_TK_BPS         # default: time-stop at e+h
    k_exit = np.full(n, h)
    rows = np.arange(n)
    m = k_sl < k_tp
    net[m]    = relf[rows[m], k_sl[m] - 1] * 1e4 - MK_TK_BPS
    k_exit[m] = k_sl[m]
    m = k_tp < k_sl
    net[m]    = tp * 1e4 - MK_MK_BPS
    k_exit[m] = k_tp[m]
    net[~filled] = np.nan
    return net, filled, k_exit, k_fill

def maker_stats(j, rate, variant='ens', wait=ENTRY_WAIT, delta_mult=ENTRY_DELTA,
                delay=ENTRY_DELAY, timing=None, tp_mult=TP_MULT, sl_mult=SL_MULT):
    # timing='h2': triggers where the h2 regression head agrees with the trade
    # direction (sgn·pred_h2 > 0 → the move is leaving NOW) enter TAKER at the
    # signal close; the rest post the maker limit into the predicted pullback.
    # run.007's IC_h2 (t=+12.9, fast-dying mean reversion) finally gets a job.
    h, side = CLS_KEYS[j]
    th  = THETA_BY_H[h]
    sgn = 1.0 if side == 'up' else -1.0
    per_fold, EF, NF, EM, n_trig, n_taker = [], [], [], [], 0, 0
    for f_, r in zip(folds, results):
        trig = fold_triggers(r, j, rate, variant)
        e = r['test_ends'][trig]
        if timing == 'h2':
            now = sgn * r['test_pred'][trig, H2_IDX] > 0
            nt, _  = sim_exit(e[now], side, h, tp_mult * th, sl_mult * th)
            nm, fm, _, _ = sim_maker(e[~now], side, h, tp_mult * th, sl_mult * th,
                                     wait, delta_mult * th, delay)
            netf   = np.concatenate([nt, nm[fm]])
            e_f    = np.concatenate([e[now], e[~now][fm]])
            e_m    = e[~now][~fm]
            n_taker += int(now.sum())
        else:
            nm, fm, _, _ = sim_maker(e, side, h, tp_mult * th, sl_mult * th,
                                     wait, delta_mult * th, delay)
            netf, e_f, e_m = nm[fm], e[fm], e[~fm]
        n_trig += len(e)
        EF.append(e_f); NF.append(netf); EM.append(e_m)
        per_fold.append(dict(
            k=f_['k'], n=len(e), n_fill=len(e_f),
            fill=len(e_f) / len(e) if len(e) else np.nan,
            hit_f=float(L_all[e_f, j].mean()) if len(e_f) else np.nan,
            sim=float(netf.mean()) if len(e_f) else np.nan))
    e_f = np.concatenate(EF); netf = np.concatenate(NF); e_m = np.concatenate(EM)
    base = float(L_all[ends_pool, j].mean())
    hit_f = float(L_all[e_f, j].mean()) if len(e_f) else np.nan
    missed = sgn * (close[e_m + h] / close[e_m] - 1.0) * 1e4
    pooled = dict(
        n=n_trig, n_fill=len(e_f), n_taker=n_taker,
        fill=len(e_f) / n_trig if n_trig else np.nan,
        hit_f=hit_f, base=base,
        lift_f=(hit_f / base) if (len(e_f) and base > 0) else np.nan,
        sim=float(netf.mean()) if len(e_f) else np.nan,
        ci=boot_ci_mean(netf, day_of(e_f)),
        ev=float(netf.sum()) / n_trig if n_trig else np.nan,
        miss_gross=float(missed.mean()) if len(e_m) else np.nan,
        hit_m=float(L_all[e_m, j].mean()) if len(e_m) else np.nan)
    return per_fold, pooled

def show_maker(j, rate, variant='ens', with_folds=False, **kw):
    pf, p = maker_stats(j, rate, variant, **kw)
    h, side = CLS_KEYS[j]
    print(f'  {side}{h:<3d} {rate*100:7.3f}%  {variant:4s}  {p["n"]:>6,}  '
          f'{_fmt(p["fill"]*100 if np.isfinite(p["fill"]) else np.nan, "5.1f")}  '
          f'{p["n_fill"]:>6,}  '
          f'{_fmt(p["hit_f"]*100 if np.isfinite(p["hit_f"]) else np.nan, "5.1f")}  '
          f'{_fmt(p["lift_f"], "5.2f")}  {_fmt(p["sim"], "+8.2f")}  '
          f'[{_fmt(p["ci"][0], "+7.2f")}, {_fmt(p["ci"][1], "+7.2f")}]  '
          f'{_fmt(p["ev"], "+7.2f")}  {_fmt(p["miss_gross"], "+7.1f")}  '
          f'{_fmt(p["hit_m"]*100 if np.isfinite(p["hit_m"]) else np.nan, "5.1f")}')
    if with_folds:
        for r_ in pf:
            print(f'        fold {r_["k"]}: n={r_["n"]:<6,} '
                  f'fill={_fmt(r_["fill"]*100 if np.isfinite(r_["fill"]) else np.nan, "5.1f")}% '
                  f'hit_f={_fmt(r_["hit_f"]*100 if np.isfinite(r_["hit_f"]) else np.nan, "5.1f")}% '
                  f'sim={_fmt(r_["sim"], "+8.2f")} bps  (n_fill={r_["n_fill"]:,})')
    return pf, p

MHDR = ('  head    rate     var   n_trig  fill%  n_fill  hit_f%  lift_f   '
        'sim bps  [95% CI, day-boot]  EV/trig   missed  hit_m%')

print(f'════ MAKER ENTRY, primary config (wait={ENTRY_WAIT} bars, '
      f'δ={ENTRY_DELTA}·θ, delay={ENTRY_DELAY}) — h{H_SEL} heads ════')
print('sim/CI = filled trades; EV/trig = per-signal, unfilled count 0;')
print('missed = gross endpoint bps of unfilled triggers, hit_m = their hit rate.')
print(MHDR)
for j in (SEL_UP_J, SEL_DN_J):
    for rate in TRIGGER_RATES:
        show_maker(j, rate, 'ens', with_folds=(rate == PRIMARY_RATE))

print('\n════ Seed-unanimity + maker entry (the 49.7%-hit heads) ════')
print(MHDR)
for j in (SEL_UP_J, SEL_DN_J):
    for rate in TRIGGER_RATES:
        show_maker(j, rate, 'unan')

print(f'\n════ h2-timed hybrid @ {PRIMARY_RATE*100:.1f}% ens: taker when the h2 '
      f'reg head says "leaving now", maker limit otherwise ════')
print(MHDR)
for j in (SEL_UP_J, SEL_DN_J):
    _, p = show_maker(j, PRIMARY_RATE, 'ens', timing='h2')
    print(f'        (routed taker: {p["n_taker"]:,}/{p["n"]:,} = '
          f'{100*p["n_taker"]/max(p["n"],1):.1f}%)')

print(f'\n════ Exploratory: wait × δ grid, h{H_SEL} @ {PRIMARY_RATE*100:.1f}% ens '
      f'(only the primary config is gated) ════')
print('  head   wait   δ·θ    fill%   hit_f%    sim bps   EV/trig')
for j in (SEL_UP_J, SEL_DN_J):
    h, side = CLS_KEYS[j]
    for w in WAIT_GRID:
        for dm in DELTA_GRID:
            _, p = maker_stats(j, PRIMARY_RATE, 'ens', wait=w, delta_mult=dm)
            print(f'  {side}{h:<4d} {w:4d}  {dm:4.2f}  '
                  f'{_fmt(p["fill"]*100 if np.isfinite(p["fill"]) else np.nan, "6.1f")}  '
                  f'{_fmt(p["hit_f"]*100 if np.isfinite(p["hit_f"]) else np.nan, "6.1f")}  '
                  f'{_fmt(p["sim"], "+9.2f")}  {_fmt(p["ev"], "+8.2f")}')

print(f'\n════ Entry-delay stress @ {PRIMARY_RATE*100:.1f}% ens (the other run.008 '
      f'promise) ════')
print('taker+1bar: enter taker at close[e+1] instead of close[e] (sim_exit on')
print('the shifted anchor, time-stop still e+h). If run.007\'s taker edge moves')
print('materially, the zero-latency assumption was doing real work.')
print('  head    variant          n     sim bps   [95% CI, day-boot]')
for j in (SEL_UP_J, SEL_DN_J):
    h, side = CLS_KEYS[j]
    th = THETA_BY_H[h]
    for tag, delay_ in (('taker+0 (run.007)', 0), ('taker+1bar', 1)):
        E, S = [], []
        for r in results:
            e = r['test_ends'][fold_triggers(r, j, PRIMARY_RATE, 'ens')]
            sn, _ = sim_exit(e + delay_, side, h - delay_, TP_MULT * th, SL_MULT * th)
            E.append(e); S.append(sn)
        e, sn = np.concatenate(E), np.concatenate(S)
        ci = boot_ci_mean(sn, day_of(e))
        print(f'  {side}{h:<4d}  {tag:16s} {len(e):>6,}  {_fmt(sn.mean() if len(e) else np.nan, "+9.2f")}  '
              f'[{_fmt(ci[0], "+7.2f")}, {_fmt(ci[1], "+7.2f")}]')
    for tag, delay_ in (('maker delay=1', 1),):
        _, p = maker_stats(j, PRIMARY_RATE, 'ens', delay=delay_)
        print(f'  {side}{h:<4d}  {tag:16s} {p["n_fill"]:>6,}  {_fmt(p["sim"], "+9.2f")}  '
              f'[{_fmt(p["ci"][0], "+7.2f")}, {_fmt(p["ci"][1], "+7.2f")}]  '
              f'(fill {_fmt(p["fill"]*100 if np.isfinite(p["fill"]) else np.nan, "4.1f")}%)')

print(f'\n════ Exploratory: all horizons, maker primary config @ '
      f'{PRIMARY_RATE*100:.1f}% ens ════')
print(MHDR)
for j, (h, s) in enumerate(CLS_KEYS):
    if h != H_SEL:
        show_maker(j, PRIMARY_RATE, 'ens')

# ── non-overlapping maker trades: one position at a time, 1 BTC ───────────
# A resting limit commits the book: unfilled triggers block new posts until
# cancel (delay+wait bars). Filled trades block until exit.
def non_overlap_maker(j, rate, variant='ens'):
    h, side = CLS_KEYS[j]
    th  = THETA_BY_H[h]
    sgn = 1.0 if side == 'up' else -1.0
    ends = np.concatenate([r['test_ends'][fold_triggers(r, j, rate, variant)]
                           for r in results])   # test ranges ordered, disjoint
    nets, fm, kex, _ = sim_maker(ends, side, h, TP_MULT * th, SL_MULT * th,
                                 ENTRY_WAIT, ENTRY_DELTA * th, ENTRY_DELAY)
    usd, busy, n_posted = [], -1, 0
    for i, e in enumerate(ends):
        if e < busy:
            continue
        n_posted += 1
        if fm[i]:
            lim = close[e] * (1.0 - sgn * ENTRY_DELTA * th)
            usd.append(lim * nets[i] / 1e4)
            busy = e + kex[i]
        else:
            busy = e + min(ENTRY_DELAY + ENTRY_WAIT, h)
    usd = np.array(usd)
    tag = f'{side}{h} @{rate*100:.1f}% {variant}'
    if len(usd) == 0:
        print(f'  {tag:24s}  {n_posted:>5,} posted   no fills')
        return
    print(f'  {tag:24s}  {n_posted:>5,} posted  {len(usd):>5,} filled   '
          f'total {usd.sum():+10.0f} $   mean {usd.mean():+7.1f} $   '
          f'win {100*(usd > 0).mean():.1f}%')

print(f'\n════ Non-overlapping 1-BTC maker trades (wait={ENTRY_WAIT}, '
      f'tp={TP_MULT}·θ, sl={SL_MULT}·θ, time-stop h) ════')
for variant in ('ens', 'unan'):
    for j in (SEL_UP_J, SEL_DN_J):
        non_overlap_maker(j, PRIMARY_RATE, variant)


In [ ]:
# ── Cell 11: GATE (pre-registered) ─────────────────────────────────────────
# At the H_SEL=6 heads (θ=20bp), primary rate 0.1%, rolling 'ens' variant,
# MAKER entry at the primary config (wait=ENTRY_WAIT, δ=ENTRY_DELTA, delay=0),
# exit TP=SL=1.0·θ anchored at the entry fill, for at least one side:
#   A. pooled maker-sim net bps (FILLED trades) > 0 AND day-bootstrap 95% CI
#      excludes 0
#   B. pooled hit lift ≥ 3× AND per-fold lift ≥ 2× in ≥ 3 of 4 folds
#      (unchanged from run.006/007 — measured on ALL triggers via cell 10's
#      trigger_stats: the ranking signal must persist, independent of fills)
#   C. fill honesty: pooled fill rate ≥ FILL_MIN AND per-fold maker sim > 0 in
#      ≥ 3 of 4 folds (a fold with zero fills counts as a fail — run.007's
#      fold-2 regime collapse must show up here, not hide in the pooled mean)
#   D. neighbouring rates (0.01%, 1%) pooled maker sim > 0
# Registered before seeing any test number (this cell was written with the
# notebook, 2026-07-10). Everything else in cells 10/10M is hypothesis-
# generating — in particular the h2-hybrid and the wait×δ grid are NOT gated.
NEIGHBOUR_RATES = [r_ for r_ in TRIGGER_RATES if r_ != PRIMARY_RATE]

def gate_side(j):
    h, side = CLS_KEYS[j]
    mpf, mp = maker_stats(j, PRIMARY_RATE, 'ens')
    tpf, tp_ = trigger_stats(j, PRIMARY_RATE, 'ens')
    A = bool(mp['n_fill'] > 0 and np.isfinite(mp['sim']) and mp['sim'] > 0
             and np.isfinite(mp['ci'][0]) and mp['ci'][0] > 0)
    ok_lift = sum(1 for r_ in tpf if np.isfinite(r_['lift']) and r_['lift'] >= 2.0)
    B = bool(np.isfinite(tp_['lift']) and tp_['lift'] >= 3.0
             and ok_lift >= N_FOLDS - 1)
    ok_sim = sum(1 for r_ in mpf if np.isfinite(r_['sim']) and r_['sim'] > 0)
    C = bool(np.isfinite(mp['fill']) and mp['fill'] >= FILL_MIN
             and ok_sim >= N_FOLDS - 1)
    neigh = [maker_stats(j, r_, 'ens')[1] for r_ in NEIGHBOUR_RATES]
    D = all(q['n_fill'] > 0 and np.isfinite(q['sim']) and q['sim'] > 0
            for q in neigh)
    print(f'  {side}{h} @ {PRIMARY_RATE*100:.1f}%:')
    print(f'    A  maker sim {_fmt(mp["sim"], "+.2f")} bps, CI '
          f'[{_fmt(mp["ci"][0], "+.2f")}, {_fmt(mp["ci"][1], "+.2f")}] '
          f'(need CI > 0)      → {"pass" if A else "fail"}')
    print(f'    B  lift {_fmt(tp_["lift"], ".2f")}x pooled (need ≥3), '
          f'folds ≥2x: {ok_lift}/{N_FOLDS} (need ≥{N_FOLDS-1})   '
          f'→ {"pass" if B else "fail"}')
    print(f'    C  fill {_fmt(mp["fill"]*100 if np.isfinite(mp["fill"]) else np.nan, ".1f")}% '
          f'(need ≥{FILL_MIN*100:.0f}%), folds sim>0: {ok_sim}/{N_FOLDS} '
          f'(need ≥{N_FOLDS-1}) → {"pass" if C else "fail"}')
    for r_, q in zip(NEIGHBOUR_RATES, neigh):
        print(f'    D  @{r_*100:6.2f}%: n_fill={q["n_fill"]:<5,} '
              f'maker sim {_fmt(q["sim"], "+.2f")} bps')
    print(f'    D  neighbours maker sim > 0                            '
          f'→ {"pass" if D else "fail"}')
    return A and B and C and D

print('GATE (pre-registered):')
up_pass = gate_side(SEL_UP_J)
dn_pass = gate_side(SEL_DN_J)
GATE_PASS = up_pass or dn_pass
print(f'\n  → GATE {"PASSED" if GATE_PASS else "FAILED"}')
if GATE_PASS:
    feat_note = ('schema-3/4 features' if V3_AVAILABLE
                 else 'the v1 feature subset (v1-compat mode — not the schema-3/4 test)')
    print(f'  Maker-entry selective triggers clear fees on {feat_note}.')
    print('  Before believing ANY P&L number: freqtrade cross-check with a')
    print('  limit-order fill model (run.005\'s lesson — its idealized EMA edge')
    print('  did not survive independent data). Then: cost-weighted BCE /')
    print('  first-touch labels (run.009 candidate), schema-3/4 rerun when')
    print('  ≥45 days of out3/out4 have accumulated.')
else:
    print('  Read the failure mode before concluding anything:')
    print('  - fill% < FILL_MIN, hit_m ≫ hit_f  → adverse selection: the limit')
    print('    only fills when the move fails; entry-price arithmetic is closed')
    print('    on v1 features. Check the h2-hybrid row — if IT is positive, the')
    print('    timing signal rescues the fills (candidate primary for run.009).')
    print('  - fill% fine, sim ≤ 0, hit_f ≈ cell-10 hit → fees are no longer')
    print('    the binding constraint, the hit rate is: wait for schema-3/4')
    print('    features (the 49.7% → 58%+ route), rerun unchanged.')
    print('  - A passes but C fails on folds     → regime-dependent (run.007')
    print('    fold-2 pattern); do not trade it; more data / regime features.')
    print('  - CI straddles 0 with few fills     → not proven; more data.')


In [ ]:
# ── Cell 12: Plots ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sweep = np.geomspace(3e-5, 3e-2, 12)

def plot_finite(ax, x, y, *a, **k):
    # skip all-NaN series (e.g. lift when the base rate is 0) — plotting them
    # leaves the log axis without data and savefig dies on autoscale
    y = np.asarray(y, dtype=float)
    m = np.isfinite(y)
    if m.any():
        ax.plot(np.asarray(x)[m], y[m], *a, **k)
        return True
    return False

ax = axes[0, 0]
any_pts = False
for j, color in ((SEL_UP_J, 'tab:green'), (SEL_DN_J, 'tab:red')):
    h, side = CLS_KEYS[j]
    sims, ends_ = [], []
    for r_ in sweep:
        _, p = trigger_stats(j, r_, 'ens')
        sims.append(p['sim']); ends_.append(p['net'])
    any_pts |= plot_finite(ax, sweep * 100, sims, '-', color=color,
                           label=f'{side}{h} TP/SL sim')
    any_pts |= plot_finite(ax, sweep * 100, ends_, ':', color=color,
                           label=f'{side}{h} hold-to-h')
ax.axhline(0, color='k', lw=0.5)
ax.axvline(PRIMARY_RATE * 100, color='gray', lw=0.5, ls=':')
if any_pts:
    ax.set_xscale('log')
ax.set_xlabel('nominal trigger rate (%)')
ax.set_ylabel('pooled net bps / signal'); ax.legend(fontsize=8)
ax.set_title(f'h{H_SEL}: simulated exit vs run.006-style endpoint (dotted)')

ax = axes[0, 1]
any_pts = False
for j, color in ((SEL_UP_J, 'tab:green'), (SEL_DN_J, 'tab:red')):
    h, side = CLS_KEYS[j]
    lifts = [trigger_stats(j, r_, 'ens')[1]['lift'] for r_ in sweep]
    any_pts |= plot_finite(ax, sweep * 100, lifts, color=color, label=f'{side}{h}')
ax.axhline(1, color='k', lw=0.5)
if any_pts:
    ax.set_xscale('log')
ax.set_xlabel('nominal trigger rate (%)')
ax.set_ylabel('hit-rate lift vs base'); ax.legend(fontsize=8)
ax.set_title('does conviction actually concentrate hits?')

ax = axes[1, 0]
sims_by_h = {'up': [], 'dn': []}
for j, (h, side) in enumerate(CLS_KEYS):
    _, p = trigger_stats(j, PRIMARY_RATE, 'ens')
    sims_by_h[side].append((h, p['sim'], p['ci']))
for side, color in (('up', 'tab:green'), ('dn', 'tab:red')):
    hs   = [v[0] for v in sims_by_h[side]]
    mu   = [v[1] for v in sims_by_h[side]]
    lo   = [v[1] - v[2][0] for v in sims_by_h[side]]
    hi   = [v[2][1] - v[1] for v in sims_by_h[side]]
    ax.errorbar(hs, mu, yerr=[lo, hi], fmt='o-', color=color, capsize=3,
                label=f'{side} heads')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('horizon h (bars of 15s)'); ax.set_ylabel('sim net bps / signal')
ax.legend(fontsize=8)
ax.set_title(f'sim net vs horizon @ {PRIMARY_RATE*100:.1f}% (day-boot 95% CI)')

ax = axes[1, 1]
step = max(len(ends_pool) // 5000, 1)
ax.plot(dt_all[ends_pool[::step]], close[ends_pool[::step]], color='0.7', lw=0.7)
for j, color, mk in ((SEL_UP_J, 'tab:green', '^'), (SEL_DN_J, 'tab:red', 'v')):
    e = np.concatenate([r['test_ends'][fold_triggers(r, j, PRIMARY_RATE, 'ens')]
                        for r in results])
    ax.scatter(dt_all[e], close[e], s=12, color=color, marker=mk,
               label=f'{CLS_KEYS[j][1]}{CLS_KEYS[j][0]} trigger', zorder=3)
ax.legend(fontsize=8)
ax.set_title(f'primary triggers ({PRIMARY_RATE*100:.1f}% nominal, rolling τ) on price')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'run008_selective.png')
plt.savefig(plot_path, dpi=110, bbox_inches='tight')
plt.show()
print(f'Saved → {plot_path}')


In [ ]:
# ── Cell 13: Save to Drive + how to read this run ──────────────────────────
import shutil

model_path = os.path.join(OUTPUT_DIR, f'lstm_run008_h{H_SEL}.pt')
torch.save({
    'model_states': results[-1]['states'],     # all seeds, last (largest-train) fold
    'scaler':       results[-1]['scaler'],
    'features':     features,
    'horizons':     HORIZONS,
    'cls_keys':     CLS_KEYS,
    'seeds':        SEEDS,
    'config':       dict(seq_len=SEQ_LEN, hidden_dim=HIDDEN_DIM,
                         num_layers=NUM_LAYERS, dropout=DROPOUT,
                         vol_window=VOL_WINDOW, target_clip=TARGET_CLIP,
                         window_sec=WINDOW_SEC, h_trade=H_TRADE, h_sel=H_SEL,
                         theta_by_h=THETA_BY_H, pos_w_cap=POS_W_CAP, bce_w=BCE_W,
                         tp_mult=TP_MULT, sl_mult=SL_MULT,
                         roll_cal_days=ROLL_CAL_DAYS, min_cal_n=MIN_CAL_N,
                         entry_wait=ENTRY_WAIT, entry_delta=ENTRY_DELTA,
                         entry_delay=ENTRY_DELAY, fill_min=FILL_MIN,
                         sample_w_base=SAMPLE_W_BASE,
                         loss_w_alpha=LOSS_W_ALPHA, loss_w_cap=LOSS_W_CAP),
    'test_ic_per_fold':      [r['test_ic'] for r in results],
    'test_ap_per_fold':      [r['test_ap'] for r in results],
    # deployment note: live thresholds come from the ROLLING rule (trailing
    # ROLL_CAL_DAYS of live probs); these frozen val quantiles only seed it.
    'primary_tau_per_fold':  [{CLS_COLS[j]: float(np.quantile(r['val_prob'][:, j],
                                                              1 - PRIMARY_RATE))
                               for j in range(len(CLS_KEYS))} for r in results],
}, model_path)
print(f'Saved model → {model_path}')

from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
for p in [scores_path, model_path, os.path.join(OUTPUT_DIR, 'run008_selective.png')]:
    if os.path.exists(p):
        shutil.copy(p, DRIVE_SAVE_DIR)
        print(f'  → Drive: {os.path.join(DRIVE_SAVE_DIR, os.path.basename(p))}')
drive.flush_and_unmount()

print('\nHow to read this run:')
print(' 1. Cell 10 must ≈ reproduce run.007 (same training bytes, same data →')
print('    same taker tables: up6 −10.0 / dn6 −12.9 bps, lift 9.4/9.8). If it')
print('    does not, stop — the data changed, nothing below is comparable.')
print(' 2. Cell 10M headline: read sim, fill% and EV/trig TOGETHER. sim > 0')
print('    with tiny fill% is not tradeable; EV/trig is the honest per-signal')
print('    number to hold against cell 10\'s taker sim column.')
print(' 3. hit_m vs hit_f is the adverse-selection verdict: hit_m ≫ hit_f =')
print('    the limit fills only when the move fails — maker entry self-defeats.')
print(' 4. Cell 11 GATE is the only pre-registered evidence (h6, 0.1%, rolling')
print('    ens, maker wait=2 δ=0, TP=SL=θ). The h2-hybrid and wait×δ grid are')
print('    hypotheses for run.009, not results.')
print(' 5. If the gate passes: freqtrade cross-check with a limit-fill model')
print('    BEFORE believing P&L (run.005\'s lesson). If it fails: cell 11')
print('    failure-mode guide — adverse selection vs hit-rate-bound matters.')
print(' 6. Schema-3/4 rerun needs ≥45 days of out3/out4 in the tar (~late Aug);')
print('    rebuild the tar with out*.csv.gz, not out.*.')
